## Notebook Summary

This notebook is dedicated to processing RxNorm RRF files from `/content/drive/MyDrive/05_data/RxNorm` and SNOMED CT data from S3.

### Actions Taken:

1.  **Google Drive Connection:** Google Drive was successfully mounted.
2.
3.  **SNOMED CT Data Download and Processing:**
    *   AWS credentials were retrieved from user data.
    *   SNOMED CT data (`SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z.zip`) was downloaded from an S3 bucket to `/content/snomed_data`.
    *   The downloaded zip file was extracted to `/content/snomed_data/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z`.
    *   The directory structure of the extracted SNOMED CT data was explored and listed.
    *   For various SNOMED CT RF2 files found in the `Full/Terminology`, `Full/Refset/Language`, `Full/Refset/Map`, `Full/Refset/Content`, `Full/Refset/Metadata`, `Snapshot/Terminology`, `Snapshot/Refset/Metadata`, `Snapshot/Refset/Language`, `Snapshot/Refset/Map`, `Snapshot/Refset/Content` subdirectories, dedicated sections were created with a markdown overview, column descriptions, and a Python code cell to read and display the first 10 rows using `pd.read_csv` (configured for tab-separated files with a header, `sep='\t', header=0, on_bad_lines='skip', nrows=10`).

### Content:

The notebook contains markdown cells explaining the purpose of each RxNorm and SNOMED CT file and Python code cells displaying the first 10 rows of the files. This structure provides a quick overview of the datasets' content and structure.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import userdata
aws_access_key_id = userdata.get('AWS_Access_Key')
aws_secret_access_key = userdata.get('AWS_Secret_Access_Key')

### Download data from S3

In [ ]:
!pip install -q boto3

import boto3
from boto3 import Session

# Your AWS creds (not from the license as they will not have rights for this location)
AWS_ACCESS_KEY_ID = aws_access_key_id
AWS_SECRET_ACCESS_KEY = aws_secret_access_key
region_name = 'us-east-1'

# MFA
mfa_code = "979769"

session = Session(aws_access_key_id=AWS_ACCESS_KEY_ID, aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
                  region_name=region_name)

client = session.client('sts')
response = client.get_session_token(
    DurationSeconds=3600,
    SerialNumber='arn:aws:iam::175460536396:mfa/Ozgur.Caglayan',
    TokenCode=mfa_code,
)
credentials = response['Credentials']
credentials

s3 = boto3.client('s3', aws_access_key_id=credentials['AccessKeyId'], aws_secret_access_key=credentials['SecretAccessKey'],
                    aws_session_token=credentials['SessionToken'])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.0 MB/s eta 0:00:00


In [ ]:
import os

!mkdir -p snomed_data # Ensure the directory exists

# The full S3 path provided: s3://source.johnsnowlabs.com/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z.zip

file_name = "SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z.zip"

BUCKET = "source.johnsnowlabs.com"

s3_file_path = file_name # The key in the S3 bucket is just the file name

destination_path = os.path.join('snomed_data', file_name)

print(f"Downloading '{file_name}' from S3 bucket '{BUCKET}'...")
s3.download_file(Bucket=BUCKET, Key=s3_file_path, Filename=destination_path)
print("Download complete.")

Download complete.


In [ ]:
import shutil
ZIPPED_FILE_PATH = "/content/snomed_data/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z.zip"
UNZIPPED_FILE = "/content/snomed_data/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z" # ZIPPED_FILE_PATH - ".zip"
shutil.unpack_archive(ZIPPED_FILE_PATH, UNZIPPED_FILE, "zip")

### Explore DIR Structure

In [ ]:
import os

snomed_data_path = "/content/snomed_data/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z"

print(f"Listing directory structure for: {snomed_data_path}\n")
!ls -R {snomed_data_path}

Listing directory structure for: /content/snomed_data/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z

/content/snomed_data/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z:
SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z

/content/snomed_data/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z:
Full  Readme_en_20260301.txt  release_package_information.json	Snapshot

/content/snomed_data/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z/Full:
Refset	Terminology

/content/snomed_data/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z/Full/Refset:
Content  Language  Map	Metadata

/content/snomed_data/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260

# SNOMED CT Data Processing

In [ ]:
import pandas as pd
import os

snomed_data_path = "/content/snomed_data/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z"
snomed_base_path = os.path.join(snomed_data_path, os.path.basename(snomed_data_path))

print(f"SNOMED CT Base Path: {snomed_base_path}")

SNOMED CT Base Path: /content/snomed_data/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z


### Full/Terminology

## sct2_Concept_Full_US1000124_20260301.txt

### Overview
The `sct2_Concept_Full` file is one of the core SNOMED CT RF2 files, containing details about all active and inactive concepts in the terminology. Each concept represents a clinical idea or meaning.

### Column Descriptions
SNOMED CT RF2 files typically follow a standard structure. For concept files, the columns usually include:
*   **`id`**: Unique identifier for the concept.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the concept is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`definitionStatusId`**: Indicates whether the concept is primitive (900000000000074008) or fully defined (900000000000073002).

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Terminology', 'sct2_Concept_Full_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    # SNOMED CT RF2 files are typically tab-separated and have a header row.
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'sct2_Concept_Full_US1000124_20260301.txt'

Successfully loaded 'sct2_Concept_Full_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,definitionStatusId
0,100005,20020131,0,900000000000207008,900000000000074008
1,101009,20020131,1,900000000000207008,900000000000074008
2,102002,20020131,1,900000000000207008,900000000000074008
3,103007,20020131,1,900000000000207008,900000000000074008
4,104001,20020131,1,900000000000207008,900000000000073002
5,105000,20020131,1,900000000000207008,900000000000074008
6,105000,20040731,0,900000000000207008,900000000000074008
7,106004,20020131,1,900000000000207008,900000000000074008
8,106004,20240301,0,900000000000207008,900000000000074008
9,107008,20020131,1,900000000000207008,900000000000074008


### Full/Refset/Language

## sct2_Description_Full-en_US1000124_20260301.txt

### Overview
The `sct2_Description_Full` file contains all textual descriptions (terms) associated with SNOMED CT concepts. This includes fully specified names, preferred terms, and synonyms for each concept, often filtered by language.

### Column Descriptions
For description files, the columns usually include:
*   **`id`**: Unique identifier for the description.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the description is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`conceptId`**: Identifier of the concept to which this description is attached.
*   **`languageCode`**: Language of the term (e.g., 'en').
*   **`typeId`**: Type of description (e.g., 900000000000003001 for Fully Specified Name, 900000000000013009 for Synonym).
*   **`term`**: The actual textual description.
*   **`caseSignificanceId`**: Indicates how the term's casing should be treated for matching (e.g., 900000000000017005 for entire term case insensitive).

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Terminology', 'sct2_Description_Full-en_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'sct2_Description_Full-en_US1000124_20260301.txt'

Successfully loaded 'sct2_Description_Full-en_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,conceptId,languageCode,typeId,term,caseSignificanceId
0,101013,20020131,1,900000000000207008,126813005,en,900000000000013009,Neoplasm of anterior aspect of epiglottis,900000000000020002
1,101013,20170731,1,900000000000207008,126813005,en,900000000000013009,Neoplasm of anterior aspect of epiglottis,900000000000448009
2,102018,20020131,1,900000000000207008,126814004,en,900000000000013009,Neoplasm of junctional region of epiglottis,900000000000020002
3,102018,20170731,1,900000000000207008,126814004,en,900000000000013009,Neoplasm of junctional region of epiglottis,900000000000448009
4,103011,20020131,1,900000000000207008,126815003,en,900000000000013009,Neoplasm of lateral wall of oropharynx,900000000000020002
5,103011,20170731,1,900000000000207008,126815003,en,900000000000013009,Neoplasm of lateral wall of oropharynx,900000000000448009
6,104017,20020131,1,900000000000207008,126816002,en,900000000000013009,Neoplasm of posterior wall of oropharynx,900000000000020002
7,104017,20170731,1,900000000000207008,126816002,en,900000000000013009,Neoplasm of posterior wall of oropharynx,900000000000448009
8,105016,20020131,1,900000000000207008,126817006,en,900000000000013009,Neoplasm of esophagus,900000000000020002
9,105016,20170731,1,900000000000207008,126817006,en,900000000000013009,Neoplasm of esophagus,900000000000448009


## sct2_Identifier_Full_US1000124_20260301.txt

### Overview
The `sct2_Identifier_Full` file contains all identifiers for SNOMED CT components (concepts, descriptions, and relationships) that are maintained within the system but may not be the primary identifiers. This is useful for tracking external identifiers or alternative identification schemes.

### Column Descriptions
For identifier files, the columns typically include:
*   **`identifierSchemeId`**: Identifier of the scheme to which the identifier belongs.
*   **`alternateIdentifier`**: The alternative identifier for the component.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the component is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`referencedComponentId`**: Identifier of the SNOMED CT component to which the identifier applies.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Terminology', 'sct2_Identifier_Full_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'sct2_Identifier_Full_US1000124_20260301.txt'

Successfully loaded 'sct2_Identifier_Full_US1000124_20260301.txt'. Displaying the first 10 rows:


,alternateIdentifier,effectiveTime,active,moduleId,identifierSchemeId,referencedComponentId


## sct2_RelationshipConcreteValues_Full_US1000124_20260301.txt

### Overview
The `sct2_RelationshipConcreteValues_Full` file contains all relationships where the target is a concrete value (e.g., a number, string, or boolean) rather than another concept. This is crucial for representing quantitative or qualitative attributes of concepts directly.

### Column Descriptions
For relationship concrete values files, the columns typically include:
*   **`id`**: Unique identifier for the relationship.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the relationship is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`sourceId`**: Identifier of the concept that is the source of the relationship.
*   **`value`**: The concrete value (e.g., '10', 'true', 'some text').
*   **`relationshipGroup`**: An integer indicating a group of relationships.
*   **`typeId`**: Identifier of the concept representing the type of relationship.
*   **`characteristicTypeId`**: Identifier of the concept representing the characteristic type of the relationship.
*   **`modifierId`**: Identifier of the concept representing the modifier of the relationship (e.g., 'existential', 'universal').

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Terminology', 'sct2_RelationshipConcreteValues_Full_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'sct2_RelationshipConcreteValues_Full_US1000124_20260301.txt'

Successfully loaded 'sct2_RelationshipConcreteValues_Full_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,sourceId,value,relationshipGroup,typeId,characteristicTypeId,modifierId
0,13830203029,20210731,1,900000000000207008,830045007,#3,0,1142139005,900000000000011006,900000000000451002
1,13830204024,20210731,1,900000000000207008,830064001,#3,0,1142139005,900000000000011006,900000000000451002
2,13830205020,20210731,1,900000000000207008,830066004,#3,0,1142139005,900000000000011006,900000000000451002
3,13830206021,20210731,1,900000000000207008,830108003,#1,0,1142139005,900000000000011006,900000000000451002
4,13830207028,20210731,1,900000000000207008,830110001,#1,0,1142139005,900000000000011006,900000000000451002
5,13830208022,20210731,1,900000000000207008,830208008,#1,0,1142139005,900000000000011006,900000000000451002
6,13830209025,20210731,1,900000000000207008,830210005,#1,0,1142139005,900000000000011006,900000000000451002
7,13830210024,20210731,1,900000000000207008,830203004,#1,0,1142139005,900000000000011006,900000000000451002
8,13830211023,20210731,1,900000000000207008,830204005,#1,0,1142139005,900000000000011006,900000000000451002
9,13830212027,20210731,1,900000000000207008,830205006,#1,0,1142139005,900000000000011006,900000000000451002


## sct2_Relationship_Full_US1000124_20260301.txt

### Overview
The `sct2_Relationship_Full` file contains all relationships between SNOMED CT concepts, defining the hierarchy and other semantic connections. These relationships are fundamental for navigating and understanding the structure of SNOMED CT.

### Column Descriptions
For relationship files, the columns typically include:
*   **`id`**: Unique identifier for the relationship.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the relationship is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`sourceId`**: Identifier of the concept that is the source of the relationship.
*   **`destinationId`**: Identifier of the concept that is the target of the relationship.
*   **`relationshipGroup`**: An integer indicating a group of relationships.
*   **`typeId`**: Identifier of the concept representing the type of relationship (e.g., 'Is a', 'Associated morphology').
*   **`characteristicTypeId`**: Identifier of the concept representing the characteristic type of the relationship (e.g., 'Inferred relationship', 'Stated relationship').
*   **`modifierId`**: Identifier of the concept representing the modifier of the relationship (e.g., 'Existential restriction', 'Universal restriction').

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Terminology', 'sct2_Relationship_Full_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'sct2_Relationship_Full_US1000124_20260301.txt'

Successfully loaded 'sct2_Relationship_Full_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,sourceId,destinationId,relationshipGroup,typeId,characteristicTypeId,modifierId
0,100022,20020131,1,900000000000207008,100000000,102272007,0,116680003,900000000000011006,900000000000451002
1,100022,20090731,0,900000000000207008,100000000,102272007,0,116680003,900000000000011006,900000000000451002
2,101021,20020131,1,900000000000207008,10000006,29857009,0,116680003,900000000000011006,900000000000451002
3,102025,20020131,1,900000000000207008,10000006,9972008,0,116680003,900000000000011006,900000000000451002
4,103024,20020131,1,900000000000207008,1000004,19130008,0,116680003,900000000000011006,900000000000451002
5,103024,20030131,0,900000000000207008,1000004,19130008,0,116680003,900000000000011006,900000000000451002
6,104029,20020131,1,900000000000207008,100001001,102272007,0,116680003,900000000000011006,900000000000451002
7,104029,20090731,0,900000000000207008,100001001,102272007,0,116680003,900000000000011006,900000000000451002
8,105028,20020131,1,900000000000207008,100002008,102272007,0,116680003,900000000000011006,900000000000451002
9,105028,20090731,0,900000000000207008,100002008,102272007,0,116680003,900000000000011006,900000000000451002


## sct2_sRefset_OWLExpressionFull_US1000124_20260301.txt

### Overview
The `sct2_sRefset_OWLExpressionFull` file contains all OWL (Web Ontology Language) expressions that define SNOMED CT concepts and their properties formally. This file is used for advanced ontological reasoning and interoperability with OWL-based systems.

### Column Descriptions
For OWL expression refset files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the reference set itself.
*   **`referencedComponentId`**: Identifier of the SNOMED CT concept to which the OWL expression applies.
*   **`owlExpression`**: The OWL expression string.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Terminology', 'sct2_sRefset_OWLExpressionFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'sct2_sRefset_OWLExpressionFull_US1000124_20260301.txt'

Successfully loaded 'sct2_sRefset_OWLExpressionFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,owlExpression
0,80001735-381a-4c86-a986-a6ebd875f6c7,20190731,1,900000000000207008,733073007,42061009,SubClassOf(:42061009 :398334008)
1,80002779-6efa-491f-88d3-8a393898bbe4,20190731,1,900000000000207008,733073007,239604004,SubClassOf(:239604004 ObjectIntersectionOf(:26...
2,80004459-1608-4ec1-9d41-ae994cd7f6a9,20190731,1,900000000000207008,733073007,283910009,SubClassOf(:283910009 :283904009)
3,80005cdc-07bf-41b8-9e90-47393b071a6a,20190731,1,900000000000207008,733073007,721657003,EquivalentClasses(:721657003 ObjectIntersectio...
4,8000d644-fed0-498d-8882-3e69d3f964d1,20190731,1,900000000000207008,733073007,87885009,SubClassOf(:87885009 :2090006)
5,80012aef-d46d-4fbb-b0b7-0e10e4f99d77,20190731,1,900000000000207008,733073007,303394007,EquivalentClasses(:303394007 ObjectIntersectio...
6,80012aef-d46d-4fbb-b0b7-0e10e4f99d77,20231201,1,900000000000207008,733073007,303394007,EquivalentClasses(:303394007 ObjectIntersectio...
7,8001566d-6275-4931-a567-f6fd7cd48bca,20190731,1,900000000000207008,733073007,84132007,SubClassOf(:84132007 :429240000)
8,80015875-73ae-4065-b2e5-c3c0e87a9214,20250301,1,900000000000207008,733073007,1144573006,EquivalentClasses(:1144573006 ObjectIntersecti...
9,80017b56-d222-4920-8585-0d34d159c755,20210930,1,900000000000207008,733073007,1162314004,EquivalentClasses(:1162314004 ObjectIntersecti...


## sct2_StatedRelationship_Full_US1000124_20260301.txt

### Overview
The `sct2_StatedRelationship_Full` file contains all explicitly asserted (stated) relationships between SNOMED CT concepts. These relationships represent the direct knowledge entered by modelers, in contrast to inferred relationships.

### Column Descriptions
For stated relationship files, the columns typically include:
*   **`id`**: Unique identifier for the relationship.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the relationship is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`sourceId`**: Identifier of the concept that is the source of the relationship.
*   **`destinationId`**: Identifier of the concept that is the target of the relationship.
*   **`relationshipGroup`**: An integer indicating a group of relationships.
*   **`typeId`**: Identifier of the concept representing the type of relationship (e.g., 'Is a', 'Associated morphology').
*   **`characteristicTypeId`**: Identifier of the concept representing the characteristic type of the relationship (always 'Stated relationship' for this file).
*   **`modifierId`**: Identifier of the concept representing the modifier of the relationship.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Terminology', 'sct2_StatedRelationship_Full_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'sct2_StatedRelationship_Full_US1000124_20260301.txt'

Successfully loaded 'sct2_StatedRelationship_Full_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,sourceId,destinationId,relationshipGroup,typeId,characteristicTypeId,modifierId
0,3187444026,20140131,1,900000000000207008,425630003,400195000,0,42752001,900000000000010007,900000000000451002
1,3187444026,20160131,0,900000000000207008,425630003,400195000,0,42752001,900000000000010007,900000000000451002
2,3192499027,20140131,0,900000000000207008,425630003,105590001,0,246075003,900000000000010007,900000000000451002
3,3574321020,20140131,1,900000000000207008,425630003,111189002,0,116680003,900000000000010007,900000000000451002
4,3574321020,20160131,0,900000000000207008,425630003,111189002,0,116680003,900000000000010007,900000000000451002
5,3829433029,20080731,1,900000000000207008,102977005,102976001,0,116680003,900000000000010007,900000000000451002
6,3829433029,20190731,0,900000000000207008,102977005,102976001,0,116680003,900000000000010007,900000000000451002
7,3829434024,20080731,1,900000000000207008,413337008,306751006,0,116680003,900000000000010007,900000000000451002
8,3829434024,20190731,0,900000000000207008,413337008,306751006,0,116680003,900000000000010007,900000000000451002
9,3829435020,20080731,1,900000000000207008,103085008,72909000,0,116680003,900000000000010007,900000000000451002


## sct2_TextDefinition_Full-en_US1000124_20260301.txt

### Overview
The `sct2_TextDefinition_Full` file contains all human-readable definitions for SNOMED CT concepts, provided as text. These definitions are essential for understanding the precise meaning of concepts within the terminology.

### Column Descriptions
For text definition files, the columns typically include:
*   **`id`**: Unique identifier for the text definition.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the text definition is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`conceptId`**: Identifier of the concept being defined.
*   **`languageCode`**: Language of the definition (e.g., 'en').
*   **`typeId`**: Type of definition (e.g., 900000000000550004 for textual definition).
*   **`term`**: The actual text of the definition.
*   **`caseSignificanceId`**: Indicates how the term's casing should be treated.
*   **`textDefinitionId`**: The identifier of the concept that represents the text definition.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Terminology', 'sct2_TextDefinition_Full-en_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'sct2_TextDefinition_Full-en_US1000124_20260301.txt'

Successfully loaded 'sct2_TextDefinition_Full-en_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,conceptId,languageCode,typeId,term,caseSignificanceId
0,2884452019,20040731,1,900000000000207008,410016009,en,900000000000550004,A decrease in lower leg circumference due to r...,900000000000017005
1,2884452019,20190731,0,900000000000207008,410016009,en,900000000000550004,A decrease in lower leg circumference due to r...,900000000000017005
2,2884453012,20050731,1,900000000000207008,416118004,en,900000000000550004,Introduction of a substance to the body,900000000000017005
3,2884454018,20030731,1,900000000000207008,125097000,en,900000000000550004,Domestic goat,900000000000017005
4,2884455017,20030731,1,900000000000207008,125099002,en,900000000000550004,Domestic sheep species,900000000000017005
5,2884455017,20110131,0,900000000000207008,125099002,en,900000000000550004,Domestic sheep species,900000000000017005
6,2884456016,20030731,1,900000000000207008,122868007,en,900000000000550004,An implantation of a staple,900000000000017005
7,2884457013,20030731,1,900000000000207008,125085001,en,900000000000550004,Equus subspecies,900000000000017005
8,2884457013,20100731,0,900000000000207008,125085001,en,900000000000550004,Equus subspecies,900000000000017005
9,2884458015,20030731,1,900000000000207008,125671007,en,900000000000550004,"Disruption of continuity of tissue, not necess...",900000000000017005


## der2_cRefset_LanguageFull-en_US1000124_20260301.txt

### Overview
The `der2_cRefset_LanguageFull` file is a Refset (Reference Set) that specifies the acceptability of descriptions (terms) within a particular language. It indicates which descriptions are preferred or acceptable for a given concept in a specific language dialect.

### Column Descriptions
For language refset files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the reference set itself (e.g., the English language reference set).
*   **`referencedComponentId`**: Identifier of the description that is a member of this refset.
*   **`acceptabilityId`**: Indicates the acceptability of the description for the language (e.g., 900000000000548007 for Preferred, 900000000000549004 for Acceptable).

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Language', 'der2_cRefset_LanguageFull-en_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_cRefset_LanguageFull-en_US1000124_20260301.txt'

Successfully loaded 'der2_cRefset_LanguageFull-en_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,acceptabilityId
0,80000517-8513-5ca0-a44c-dc66f3c3a1c6,20080731,1,900000000000207008,900000000000508004,2743026013,900000000000548007
1,80000755-c5d9-5bd8-bb64-ab8236d240d7,20020131,1,900000000000207008,900000000000509007,2320010,900000000000548007
2,8000095c-e40d-56d2-9432-7f9a716d60d2,20020131,1,900000000000207008,900000000000509007,99175018,900000000000548007
3,80000cf0-bdc8-48e9-b0e1-914e28484bbc,20210731,1,900000000000207008,900000000000508004,4571203018,900000000000549004
4,800012f3-4937-481b-b37c-7862b7073648,20220731,1,900000000000207008,900000000000509007,5071815010,900000000000548007
5,80001355-118d-5beb-a603-48dae9011f64,20020131,1,900000000000207008,900000000000508004,306538015,900000000000549004
6,800016a3-27ca-4e27-9a46-6899408ff2ce,20170131,1,900000000000207008,900000000000508004,3329492019,900000000000549004
7,800023dc-3835-5c28-8a24-3885e165e266,20020131,1,900000000000207008,900000000000509007,381144012,900000000000548007
8,80003e21-1a84-5850-b725-e1a92c6d1fd6,20020131,0,900000000000207008,900000000000509007,108667019,900000000000549004
9,80003eba-0c7c-5dd4-9f9e-a416967cbf05,20150131,1,900000000000207008,900000000000509007,3011884016,900000000000549004


### Full/Refset/Map

## der2_iisssccRefset_ExtendedMapFull_US1000124_20260301.txt

### Overview
The `der2_iisssccRefset_ExtendedMapFull` file is an extended map reference set, used for mapping SNOMED CT concepts to concepts in other terminologies or external classifications with additional attributes. It provides more detailed mapping information than a simple map refset.

### Column Descriptions
For extended map refset files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the reference set itself.
*   **`referencedComponentId`**: Identifier of the SNOMED CT concept being mapped.
*   **`mapGroup`**: A group number indicating related mapping entries.
*   **`mapPriority`**: An integer indicating the priority of the map target within a map group.
*   **`mapRule`**: A rule that defines the conditions under which a map target is applicable.
*   **`mapAdvice`**: Text providing advice or instruction about the use of the map.
*   **`mapTarget`**: The identifier of the target concept in the external terminology.
*   **`correlationId`**: An identifier representing the degree of correlation between source and target concepts.
*   **`mapCategoryId`**: An identifier for the category of the map (e.g., fully specified, partial, ambiguous).

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Map', 'der2_iisssccRefset_ExtendedMapFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_iisssccRefset_ExtendedMapFull_US1000124_20260301.txt'

Successfully loaded 'der2_iisssccRefset_ExtendedMapFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,mapGroup,mapPriority,mapRule,mapAdvice,mapTarget,correlationId,mapCategoryId
0,80004cdf-a114-59c8-88a2-b6f0283acf4f,20180301,1,5991000124107,6011000124106,128041000119107,1,1,IFA 721617001 | Primary adenocarcinoma of lowe...,IF PRIMARY ADENOCARCINOMA OF LOWER THIRD OF ES...,K22.70,447561005,447639009
1,80005aeb-477c-53dc-9a5c-ce723ca264cb,20150731,1,449080006,447562003,254153009,1,1,TRUE,ALWAYS Q79.8,Q79.8,447561005,447637006
2,80007b64-5a60-5556-88ce-22ef540a3ea5,20180301,1,5991000124107,6011000124106,281801000009108,2,2,OTHERWISE TRUE,ALWAYS B96.89,B96.89,447561005,447637006
3,80007b64-5a60-5556-88ce-22ef540a3ea5,20230901,1,5991000124107,6011000124106,281801000009108,2,10,OTHERWISE TRUE,ALWAYS B96.89,B96.89,447561005,447637006
4,80007b64-5a60-5556-88ce-22ef540a3ea5,20250301,0,5991000124107,6011000124106,281801000009108,2,10,OTHERWISE TRUE,ALWAYS B96.89,B96.89,447561005,447637006
5,80007e6a-7408-5b87-a1ec-70b212811410,20190731,1,449080006,447562003,16623961000119100,1,1,TRUE,ALWAYS D61.1,D61.1,447561005,447637006
6,80009454-5531-5f78-b7c9-d288f2346d83,20150731,1,449080006,447562003,301327002,1,1,TRUE,MAP SOURCE CONCEPT CANNOT BE CLASSIFIED WITH A...,NaN,447561005,447638001
7,80009454-5531-5f78-b7c9-d288f2346d83,20190131,0,449080006,447562003,301327002,1,1,TRUE,MAP SOURCE CONCEPT CANNOT BE CLASSIFIED WITH A...,NaN,447561005,447638001
8,8000a5af-6962-5385-9227-4038d1f7b237,20150731,1,449080006,447562003,246951001,1,1,TRUE,ALWAYS H10.8,H10.8,447561005,447637006
9,8000ab4e-1417-5c6e-bb3a-9b7ae751bc03,20250301,1,5991000124107,6011000124106,10996811000119103,1,2,IFA 53911000087103 | History of bilateral inde...,IF HISTORY OF BILATERAL INDEX FINGER AMPUTATIO...,Z89.021,447561005,447639009


## der2_sRefset_SimpleMapFull_US1000124_20260301.txt

### Overview
The `der2_sRefset_SimpleMapFull` file is a simple map reference set. It provides straightforward one-to-one mappings from SNOMED CT concepts to concepts in other terminologies or classifications without additional complex rules or attributes.

### Column Descriptions
For simple map refset files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the reference set itself.
*   **`referencedComponentId`**: Identifier of the SNOMED CT concept being mapped.
*   **`mapTarget`**: The identifier of the target concept in the external terminology.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Map', 'der2_sRefset_SimpleMapFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_sRefset_SimpleMapFull_US1000124_20260301.txt'

Successfully loaded 'der2_sRefset_SimpleMapFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,mapTarget
0,80001267-7451-550a-82f0-92cc3bdfe890,20020131,1,900000000000207008,900000000000497000,154938001,.E4D4
1,80001782-d79c-5b33-8679-c0c62beef6da,20020131,1,900000000000207008,900000000000497000,138614002,.13gX
2,8000241a-ed32-4339-876b-05fee677bda3,20180131,1,900000000000207008,900000000000497000,735755000,XUyL5
3,80002a2a-412f-59a4-b07c-6f194709c556,20020131,1,900000000000207008,900000000000497000,238194001,X40Ze
4,80004caa-f9ed-5ef8-a9fb-6c9e89e0b89d,20020131,1,900000000000207008,900000000000497000,181522009,7N72Y
5,8000501c-e5f1-5df2-91b8-d2360661e55c,20050731,1,900000000000207008,900000000000497000,416460006,XUd3b
6,800061cc-1d06-4048-874d-2d5eff76a653,20180131,1,900000000000207008,900000000000497000,16064651000119108,XUyFc
7,80009e45-0c82-4285-9ba4-b6de7e29d7e9,20190731,1,900000000000207008,900000000000497000,10823771000119109,XVAEl
8,8000b9de-9288-5bfb-905b-adb911e6cef5,20020131,1,900000000000207008,900000000000497000,317571000,za1CK
9,8000baeb-e1a1-50e8-b28f-7a687d3cf1c5,20020131,1,900000000000207008,446608001,2681003,C47.5


### Full/Refset/Content

## der2_cciRefset_RefsetDescriptorSnapshot_US1000124_20260301.txt

### Overview
The `der2_cciRefset_RefsetDescriptorSnapshot` file is the snapshot version of the Refset Descriptor file. It provides the most current metadata about reference sets, including their types, referencable components, and attributes. This file is essential for understanding the current structure and purpose of other refsets in a snapshot release.

### Column Descriptions
For refset descriptor snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the refset descriptor member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset descriptor is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the refset being described.
*   **`referencedComponentId`**: Identifier of the component that is typically referenced by this refset (e.g., a Concept ID).
*   **`attributeDescription`**: An identifier for a concept that describes the nature of the attribute within the refset.
*   **`attributeType`**: The data type of the attribute (e.g., string, integer, boolean).
*   **`attributeCardinality`**: The cardinality of the attribute (e.g., 0..1, 1..1).
*   **`attributeDataType`**: An identifier for the concept that represents the data type of the attribute.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Refset', 'Metadata', 'der2_cciRefset_RefsetDescriptorSnapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_cciRefset_RefsetDescriptorSnapshot_US1000124_20260301.txt'

Successfully loaded 'der2_cciRefset_RefsetDescriptorSnapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,attributeDescription,attributeType,attributeOrder
0,81ee191b-56fd-4950-9e14-6cb4c5cb5b1b,20170731,1,900000000000012004,900000000000456007,723561005,723574004,900000000000461009,6
1,8561568a-c56c-4b02-aaff-05e581606ebf,20140131,1,900000000000207008,900000000000456007,900000000000531004,900000000000533001,900000000000460005,1
2,85cc1c27-3a9f-5859-a1c5-7522b51ed528,20130731,1,449080006,900000000000456007,447562003,900000000000502003,900000000000478000,2
3,85d2cadc-65a0-5e0e-aa32-91236adfb1be,20020131,1,900000000000207008,900000000000456007,900000000000488004,449608002,900000000000463007,0
4,885e163a-5c99-423d-9879-dc9e10599fb5,20220131,0,900000000000207008,900000000000456007,900000000000496009,900000000000499002,900000000000465000,1
5,89613c1d-fff1-4744-951a-9c1ee341af5a,20170731,1,900000000000207008,900000000000456007,447258008,447255006,900000000000478000,1
6,89aa39de-d0a8-5879-9ea6-79a88e6d65de,20020131,1,900000000000207008,900000000000456007,900000000000526001,900000000000532006,900000000000460005,0
7,8af233ae-d139-4335-8bef-336862b68e89,20210131,1,900000000000012004,900000000000456007,1119435002,733616009,1119461003,2
8,8bc8366e-7cf1-55ef-9f90-872183466155,20020131,1,900000000000207008,900000000000456007,900000000000524003,900000000000533001,900000000000460005,1
9,8df7ee2c-73ff-4003-bc73-be305de6aae9,20170731,1,900000000000012004,900000000000456007,723604009,723572000,900000000000478000,2


## der2_ciRefset_DescriptionTypeSnapshot_US1000124_20260301.txt

### Overview
The `der2_ciRefset_DescriptionTypeSnapshot` file is the snapshot version of the Description Type file. It contains the most current details about different types of descriptions used in SNOMED CT, including their properties and characteristics. This file is used to manage and categorize textual expressions of concepts in the current state of the terminology.

### Column Descriptions
For description type refset snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the reference set itself.
*   **`referencedComponentId`**: Identifier of the concept representing the description type.
*   **`descriptionLength`**: The maximum length of the description string.
*   **`descriptionFormat`**: The format of the description (e.g., plain text, HTML).
*   **`characteristicTypeId`**: The concept ID for the characteristic type (e.g., case significance).
*   **`dataTypeId`**: The concept ID for the data type of the description type's attribute.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Refset', 'Metadata', 'der2_ciRefset_DescriptionTypeSnapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_ciRefset_DescriptionTypeSnapshot_US1000124_20260301.txt'

Successfully loaded 'der2_ciRefset_DescriptionTypeSnapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,descriptionFormat,descriptionLength
0,807f775b-1d66-5069-b58e-a37ace985dcf,20140131,1,900000000000207008,900000000000538005,900000000000550004,900000000000540000,4096
1,909a711e-b114-5543-841e-242aaa246363,20020131,1,900000000000207008,900000000000538005,900000000000013009,900000000000540000,255
2,0f928c01-b245-5907-9758-a46cbeed2674,20020131,1,900000000000207008,900000000000538005,900000000000003001,900000000000540000,255


## der2_cissccRefset_MRCMAttributeDomainSnapshot_US1000124_20260301.txt

### Overview
The `der2_cissccRefset_MRCMAttributeDomainSnapshot` file is the snapshot version of the MRCM Attribute Domain file. It contains the most current definitions of attribute domains, specifying which concepts can have which attributes and their allowed values. This file helps ensure consistency and validity in SNOMED CT content modeling for the current release.

### Column Descriptions
For MRCM attribute domain snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the MRCM attribute domain reference set.
*   **`referencedComponentId`**: Identifier of the concept representing the attribute.
*   **`domainId`**: The identifier of the concept that serves as the domain for this attribute.
*   **`grouped`**: Boolean indicating if the attribute is grouped.
*   **`attributeCardinality`**: Specifies the allowed cardinality of the attribute (e.g., 0..1, 1..*).
*   **`attributeInGroupCardinality`**: Specifies the allowed cardinality of the attribute within a group.
*   **`ruleStrengthId`**: The strength of the rule (e.g., 'mandatory', 'optional').
*   **`contentTypeId`**: The type of content that the attribute applies to.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Refset', 'Metadata', 'der2_cissccRefset_MRCMAttributeDomainSnapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_cissccRefset_MRCMAttributeDomainSnapshot_US1000124_20260301.txt'

Successfully loaded 'der2_cissccRefset_MRCMAttributeDomainSnapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,domainId,grouped,attributeCardinality,attributeInGroupCardinality,ruleStrengthId,contentTypeId
0,8038688b-80f5-49b3-b3d8-c6fe13f8fb1e,20170731,1,900000000000012004,723561005,370129005,386053000,1,0..*,0..1,723597001,723596005
1,81da6aa5-0cb1-463f-88ad-c25a795a211b,20170731,1,900000000000012004,723561005,363713009,404684003,1,0..*,0..1,723597001,723596005
2,81eded9f-9769-e314-ca69-3b7e9e49c148,20250201,1,900000000000012004,723561005,116688005,71388002,1,0..*,0..1,723597001,723596005
3,840861c7-8d9b-44a6-acfd-01f1b994b6d9,20170731,1,900000000000012004,723561005,405813007,71388002,1,0..*,0..1,723597001,723596005
4,87e54647-dee0-4bd4-8b45-0ab0fa9fad74,20210731,1,900000000000012004,723561005,1148965004,260787004,0,0..1,0..0,723597001,723596005
5,883bd314-23b7-4ccc-87d6-8067d98b3660,20180131,1,900000000000012004,723561005,704324001,363787002,1,0..*,0..1,723597001,723596005
6,89d0a100-7140-4913-bf80-bef3bc3d8495,20200731,1,900000000000012004,723561005,860779006,373873005,1,0..*,0..*,723597001,723596005
7,8ce25c12-3cc5-4ab0-84e2-fdc0a75dffed,20180131,1,900000000000012004,723561005,704327008,363787002,1,0..*,0..1,723597001,723596005
8,8d066b27-118d-4a84-886a-14dabceba155,20220731,1,900000000000012004,723561005,363698007,404684003,1,0..*,0..1,723597001,723596005
9,9339edc2-b5f1-4e6f-9fe3-d5c09f403f4d,20210731,1,900000000000012004,723561005,1148967007,260787004,0,0..1,0..0,723597001,723596005


## der2_cRefset_MRCMModuleScopeSnapshot_US1000124_20260301.txt

### Overview
The `der2_cRefset_MRCMModuleScopeSnapshot` file is the snapshot version of the MRCM Module Scope file. It defines the current scope of Machine Readable Concept Model (MRCM) rules for specific modules, indicating which MRCM rules apply to which SNOMED CT modules in the current release. This allows for effective modular content modeling.

### Column Descriptions
For MRCM module scope snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the MRCM module scope reference set.
*   **`referencedComponentId`**: Identifier of the module to which the MRCM rules apply.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Refset', 'Metadata', 'der2_cRefset_MRCMModuleScopeSnapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_cRefset_MRCMModuleScopeSnapshot_US1000124_20260301.txt'

Successfully loaded 'der2_cRefset_MRCMModuleScopeSnapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,mrcmRuleRefsetId
0,8e5766bc-7755-45fb-99c8-8d2c52e45da5,20170731,1,900000000000012004,723563008,900000000000207008,723562003
1,d3b7a01f-6fcd-4d8d-815b-ca6a63511310,20170731,1,900000000000012004,723563008,900000000000207008,723561005
2,60dcf17f-6054-4ee2-a657-606104c49bb5,20170731,1,900000000000012004,723563008,900000000000207008,723560006


## der2_scsRefset_ComponentAnnotationStringValueSnapshot_US1000124_20260301.txt

### Overview
The `der2_scsRefset_ComponentAnnotationStringValueSnapshot` file is the snapshot version of the Component Annotation String Value file. It provides the most current string-based annotations for SNOMED CT components. These annotations are used to attach arbitrary text values, such as editorial notes or comments, to concepts, descriptions, or relationships in the current release.

### Column Descriptions
For component annotation string value snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the annotation reference set.
*   **`referencedComponentId`**: Identifier of the SNOMED CT component being annotated.
*   **`stringAttributeValue`**: The string value of the annotation.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Refset', 'Metadata', 'der2_scsRefset_ComponentAnnotationStringValueSnapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_scsRefset_ComponentAnnotationStringValueSnapshot_US1000124_20260301.txt'

Successfully loaded 'der2_scsRefset_ComponentAnnotationStringValueSnapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,languageDialectCode,typeId,value
0,8000f421-3619-4607-8573-b7249677262c,20250201,1,900000000000207008,1292992004,770909004,en,1295448001,Inserm Orphanet
1,8017e26c-0999-40fa-a4ca-dfa88fe619f3,20250101,1,900000000000207008,1292992004,1222791008,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
2,8019e6c1-1612-4af1-b43e-3a49f5888214,20250101,1,900000000000207008,1292992004,1229852009,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
3,8028e13a-2b8c-4628-a588-607049b96468,20250101,1,900000000000207008,1292992004,1229851002,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
4,803dcbe9-3afc-4ef1-aebf-7ca4fbf432cc,20250101,1,900000000000207008,1292992004,719271000,en,1295448001,Inserm Orphanet
5,803ffc1d-4bfb-4d32-9a60-0c027cc858ae,20250201,1,900000000000207008,1292992004,721888002,en,1295448001,Inserm Orphanet
6,80431efd-ff00-476e-9656-b4f4aca971d2,20250201,1,900000000000207008,1292992004,770788000,en,1295448001,Inserm Orphanet
7,80446dc2-4094-454a-a7e5-f74ca5d943b8,20251001,1,900000000000207008,1292992004,1373339000,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
8,805366b6-854d-4003-900b-58f1835e9375,20250101,1,900000000000207008,1292992004,1229901006,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
9,805c83bf-747d-4a80-b4d6-217c35823a1f,20250201,1,900000000000207008,1292992004,725417001,en,1295448001,Inserm Orphanet


## der2_ssccRefset_MRCMAttributeRangeSnapshot_US1000124_20260301.txt

### Overview
The `der2_ssccRefset_MRCMAttributeRangeSnapshot` file is the snapshot version of the MRCM Attribute Range file. It specifies the valid range of values for attributes within the Machine Readable Concept Model (MRCM) for the current release. This ensures that attribute values conform to predefined constraints, contributing to the quality and consistency of SNOMED CT content.

### Column Descriptions
For MRCM attribute range snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the MRCM attribute range reference set.
*   **`referencedComponentId`**: Identifier of the attribute for which the range is being defined.
*   **`rangeConstraint`**: The constraint defining the allowed range of values.
*   **`attributeRule`**: A rule that further specifies how the range applies.
*   **`ruleStrengthId`**: The strength of the rule.
*   **`contentTypeId`**: The type of content that the attribute applies to.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Refset', 'Metadata', 'der2_ssccRefset_MRCMAttributeRangeSnapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_ssccRefset_MRCMAttributeRangeSnapshot_US1000124_20260301.txt'

Successfully loaded 'der2_ssccRefset_MRCMAttributeRangeSnapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,rangeConstraint,attributeRule,ruleStrengthId,contentTypeId
0,81288567-57a9-49b1-b7f0-bf5979a6d289,20170731,1,900000000000012004,723562003,405814001,<< 442083009 |Anatomical or acquired body stru...,<< 71388002 |Procedure (procedure)|: [0..*] { ...,723597001,723596005
1,8163e3c0-b0c4-4c92-82e7-e93a23770b17,20230630,1,900000000000012004,723562003,246090004,<< 272379006 |Event (event)| OR << 363787002 |...,<< 413350009 |Finding with explicit context (s...,723597001,723595009
2,81f6d23b-66c7-46ce-b003-08380b9b236a,20200731,1,900000000000012004,723562003,118170007,<< 125676002 |Person (person)| OR << 133928008...,<< 123038009 |Specimen (specimen)|: [0..*] { [...,723597001,723596005
3,8a4b2de3-41cf-49a4-995e-9f80c71e0684,20200731,1,900000000000012004,723562003,732943007,< 105590001 |Substance (substance)|,<< 373873005 |Pharmaceutical / biologic produc...,723597001,723596005
4,903d4612-9e2e-4434-94e8-eaff214f4467,20170731,1,900000000000012004,723562003,425391005,<< 49062001 |Device (physical object)|,<< 71388002 |Procedure (procedure)|: [0..*] { ...,723597001,723596005
5,94605782-de4e-4008-a074-39091e81d223,20210731,0,900000000000012004,723562003,766953001,< 260299005 |Number (qualifier value)|,<< 373873005 |Pharmaceutical / biologic produc...,723597001,723596005
6,96d077d2-d331-4fb6-bf38-474bd0cdfc09,20210731,1,900000000000012004,723562003,1149367008,< 27821000087106 |Product target population (q...,<< 373873005 |Pharmaceutical / biologic produc...,723597001,723596005
7,9883dd72-0fdc-47bf-89d0-4f206ca7800c,20210731,1,900000000000012004,723562003,363713009,<< 260245000 |Finding value (qualifier value)|...,<< 404684003 |Clinical finding (finding)|: [0....,723597001,723596005
8,98984674-6f17-40fe-8859-5ad2d0f5c09d,20200731,1,900000000000012004,723562003,419066007,<< 419358007 |Subject of record or other provi...,<< 404684003 |Clinical finding (finding)|: [0....,723597001,723596005
9,9ba43854-0226-48fd-baac-bfe6f19db20a,20200731,1,900000000000012004,723562003,733932009,<< 123037004 |Body structure (body structure)|,<< 123037004 |Body structure (body structure)|...,723597001,723594008


## der2_sscsRefset_MemberAnnotationStringValueSnapshot_US1000124_20260301.txt

### Overview
The `der2_sscsRefset_MemberAnnotationStringValueSnapshot` file is the snapshot version of the Member Annotation String Value file. It allows for attaching string-based annotations to reference set members in the current release. This is useful for providing additional descriptive information, comments, or provenance details about individual entries within a reference set.

### Column Descriptions
For member annotation string value snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the annotation reference set.
*   **`referencedComponentId`**: Identifier of the reference set member being annotated.
*   **`stringAttributeValue`**: The string value of the annotation.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Refset', 'Metadata', 'der2_sscsRefset_MemberAnnotationStringValueSnapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_sscsRefset_MemberAnnotationStringValueSnapshot_US1000124_20260301.txt'

Successfully loaded 'der2_sscsRefset_MemberAnnotationStringValueSnapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,referencedMemberId,languageDialectCode,typeId,value


## der2_ssRefset_ModuleDependencySnapshot_US1000124_20260301.txt

### Overview
The `der2_ssRefset_ModuleDependencySnapshot` file is the snapshot version of the Module Dependency file. It specifies dependencies between SNOMED CT modules in the current release. It records which modules depend on other modules, which is essential for managing the modular release and extension of SNOMED CT content.

### Column Descriptions
For module dependency snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the module dependency reference set.
*   **`referencedComponentId`**: Identifier of the dependent module.
*   **`sourceEffectiveTime`**: The effective time of the source module.
*   **`targetEffectiveTime`**: The effective time of the target module.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Refset', 'Metadata', 'der2_ssRefset_ModuleDependencySnapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_ssRefset_ModuleDependencySnapshot_US1000124_20260301.txt'

Successfully loaded 'der2_ssRefset_ModuleDependencySnapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,sourceEffectiveTime,targetEffectiveTime
0,9200c4da-3189-4dd5-9879-02c9ae118747,20260301,1,5991000124107,900000000000534007,900000000000207008,20260301,20260101
1,ed482ae6-dff1-42d8-8e39-353b78a60471,20260301,1,5991000124107,900000000000534007,900000000000012004,20260301,20260101
2,f6431457-161b-5b46-9217-573c20c00070,20260101,1,449080006,900000000000534007,900000000000012004,20260101,20260101
3,1244116f-fdb5-5645-afcc-5281288409da,20260101,1,900000000000207008,900000000000534007,900000000000012004,20260101,20260101
4,53773521-fd81-5dde-9a2d-8b87e8803251,20260301,1,5991000124107,900000000000534007,731000124108,20260301,20260301
5,686fe392-d2f7-586d-98aa-078aba426832,20260101,1,449080006,900000000000534007,900000000000207008,20260101,20260101
6,70edb2ac-8a7a-5a94-b385-f8ad80db3858,20260301,1,731000124108,900000000000534007,900000000000012004,20260301,20260101
7,71ae8ceb-7984-550c-b1f5-9a69681b383a,20260301,1,731000124108,900000000000534007,900000000000207008,20260301,20260101


## der2_sssssssRefset_MRCMDomainSnapshot_US1000124_20260301.txt

### Overview
The `der2_sssssssRefset_MRCMDomainSnapshot` file is the snapshot version of the MRCM Domain file. It defines the current domain of concepts that are constrained by a particular MRCM rule. It specifies which subsets of SNOMED CT concepts are subject to certain modeling rules in the current release, ensuring adherence to the established concept model.

### Column Descriptions
For MRCM domain snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the MRCM domain reference set.
*   **`referencedComponentId`**: Identifier of the concept representing the domain.
*   **`domainConstraint`**: The constraint that defines the domain.
*   **`parentDomain`**: The identifier of the parent domain, if applicable.
*   **`proximalPrimitiveConcept`**: The identifier of the closest primitive concept.
*   **`proximalPrimitiveRefinement`**: The refinement expression for the proximal primitive concept.
*   **`domainTemplateForPrecoordination`**: A template for precoordinated concepts.
*   **`domainTemplateForPostcoordination`**: A template for postcoordinated concepts.
*   **`guideURL`**: A URL to a guide or documentation related to the domain.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Refset', 'Metadata', 'der2_sssssssRefset_MRCMDomainSnapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_sssssssRefset_MRCMDomainSnapshot_US1000124_20260301.txt'

Successfully loaded 'der2_sssssssRefset_MRCMDomainSnapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,domainConstraint,parentDomain,proximalPrimitiveConstraint,proximalPrimitiveRefinement,domainTemplateForPrecoordination,domainTemplateForPostcoordination,guideURL
0,821b5d55-2985-4293-98a9-2959a69b0b3b,20250201,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
1,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20250501,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [[+id(<< 1292650...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000
2,eb0bebd1-991a-4f69-97ab-e1c5bf64dd27,20220731,1,900000000000012004,723560006,723264001,^ 723264001 |Lateralizable body structure refe...,91723000 |Anatomical structure (body structure)|,^ 723264001 |Lateralizable body structure refe...,NaN,[[+id(^ 723264001 |Lateralizable body structur...,[[+scg(^ 723264001 |Lateralizable body structu...,http://snomed.org/dom723264001
3,f1136b12-f2bd-46db-b9fb-be484129e40c,20230228,1,900000000000012004,723560006,404684003,<< 404684003 |Clinical finding (finding)|,NaN,<< 404684003 |Clinical finding (finding)|,NaN,[[+id(<< 404684003 |Clinical finding (finding)...,[[+scg(<< 404684003 |Clinical finding (finding...,http://snomed.org/dom404684003
4,07225f48-d874-403f-a051-d0fa9d2895e2,20230630,1,900000000000012004,723560006,129125009,<< 129125009 |Procedure with explicit context ...,243796009 |Situation with explicit context (si...,<< 243796009 |Situation with explicit context ...,[[1..*]] 363589002 |Associated procedure (attr...,[[+id(<< 243796009 |Situation with explicit co...,[[+scg(<< 243796009 |Situation with explicit c...,http://snomed.org/dom129125009
5,0867d3ba-1849-4de1-9812-dedcd18a561c,20230901,1,900000000000012004,723560006,781405001,<< 781405001 |Medicinal product package (produ...,373873005 |Pharmaceutical / biologic product (...,<< 781405001 |Medicinal product package (produ...,NaN,[[+id(<< 781405001 |Medicinal product package ...,[[+scg(<< 781405001 |Medicinal product package...,http://snomed.org/dom781405001
6,0b0b9db5-3be5-44e5-86e8-1389756ec245,20230630,1,900000000000012004,723560006,413350009,<< 413350009 |Finding with explicit context (s...,243796009 |Situation with explicit context (si...,<< 243796009 |Situation with explicit context ...,[[1..*]] 246090004 |Associated finding (attrib...,[[+id(<< 243796009 |Situation with explicit co...,[[+scg(<< 243796009 |Situation with explicit c...,http://snomed.org/dom413350009
7,0c1fd571-0112-4158-a41f-dae1eb98b53a,20240501,0,900000000000012004,723560006,1285465008,<< 1285465008 |Administration via specific rou...,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [[+id(<< 1294450...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom1285465008
8,0ffa1930-2a8d-4ff5-b806-f6840e8f88f3,20200731,1,900000000000012004,723560006,736542009,<< 736542009 |Pharmaceutical dose form (dose f...,NaN,<< 736542009 |Pharmaceutical dose form (dose f...,NaN,[[+id(<< 736542009 |Pharmaceutical dose form (...,[[+scg(<< 736542009 |Pharmaceutical dose form ...,http://snomed.org/dom736542009
9,125680f7-e0a6-48a9-9dd4-ffda34fcdcc9,20190731,1,900000000000012004,723560006,736478001,<< 736478001 |Basic dose form (basic dose form)|,NaN,<< 736478001 |Basic dose form (basic dose form)|,NaN,[[+id(<< 736478001 |Basic dose form (basic dos...,[[+scg(<< 736478001 |Basic dose form (basic do...,http://snomed.org/dom736478001


## der2_cRefset_AssociationFull_US1000124_20260301.txt

### Overview
The `der2_cRefset_AssociationFull` file is a concrete domain reference set member, which provides relationships between concepts or between concepts and values (e.g., numerical values). These associations add specific, detailed characteristics to SNOMED CT concepts.

### Column Descriptions
For association refset files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the reference set itself.
*   **`referencedComponentId`**: Identifier of the SNOMED CT concept to which the association applies.
*   **`targetComponentId`**: Identifier of the target concept or value in the association.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Content', 'der2_cRefset_AssociationFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_cRefset_AssociationFull_US1000124_20260301.txt'

Successfully loaded 'der2_cRefset_AssociationFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,targetComponentId
0,80001d7e-b1b9-56ac-9768-308cabe31117,20040731,1,900000000000207008,900000000000527005,290170004,216464004
1,80006a57-2ab6-524d-8cb5-ab051fb10aaa,20020131,1,900000000000207008,900000000000527005,165811000,313476009
2,8001aeda-719f-5d07-a4aa-00b734b748da,20020731,1,900000000000207008,900000000000523009,266866008,159972006
3,8001b97b-6cb7-41e4-b3d4-eb823a0559a6,20230531,1,900000000000207008,900000000000523009,246531008,128612007
4,80024464-d5ea-4a12-b933-606a6056c06f,20170731,1,900000000000207008,734138000,280164007,729256003
5,80029c0f-4be7-40c6-ac40-f707251b1f07,20241001,1,900000000000207008,900000000000526001,404148006,109969005
6,8002dce0-0f88-48f4-af67-5e8626a427fb,20210131,1,900000000000207008,900000000000523009,119310001,1003709001
7,80030572-b3de-5f0c-a1c6-86a31ca0b807,20090131,1,900000000000207008,900000000000524003,316645000,370137002
8,80033ca4-4877-4b13-b6fe-11f1604685de,20190131,1,900000000000207008,900000000000523009,14255005,46799006
9,80033dfe-45a5-579c-b6da-9c5035fca20c,20020131,1,900000000000207008,900000000000527005,138488004,315509001


## der2_cRefset_AttributeValueFull_US1000124_20260301.txt

### Overview
The `der2_cRefset_AttributeValueFull` file is a concrete domain reference set member, which contains attribute values for specific SNOMED CT concepts. This allows for the storage of non-hierarchical, descriptive information about concepts, such as clinical findings, procedures, or substances.

### Column Descriptions
For attribute value refset files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the reference set itself.
*   **`referencedComponentId`**: Identifier of the SNOMED CT concept to which the attribute value applies.
*   **`valueId`**: Identifier of the concept representing the attribute's value. In some cases, this could be a direct numerical or string value rather than a concept ID, depending on the attribute type.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Content', 'der2_cRefset_AttributeValueFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_cRefset_AttributeValueFull_US1000124_20260301.txt'

Successfully loaded 'der2_cRefset_AttributeValueFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,valueId
0,80005357-6c6d-4f02-9532-67c73e3eccc7,20190131,1,900000000000207008,900000000000490003,3672851010,723277005
1,80005fab-e950-433c-88ef-1575d3cf4de5,20221130,1,900000000000207008,900000000000490003,391438018,723277005
2,80006347-0187-5de0-9c84-ea1ddf9e3986,20020131,1,900000000000207008,900000000000489007,193284003,900000000000482003
3,800089cc-3dd8-4840-a613-99e1b3a4b6ee,20180731,1,900000000000207008,900000000000490003,3567548019,723277005
4,80009975-8e9d-5238-8e5b-146db8b24013,20020131,1,900000000000207008,900000000000490003,777428015,900000000000495008
5,80009975-8e9d-5238-8e5b-146db8b24013,20080731,0,900000000000207008,900000000000490003,777428015,900000000000495008
6,80009cd8-8d1d-414b-99e3-48d909b07812,20180731,1,900000000000207008,900000000000490003,2835304018,723277005
7,8000ba3d-f802-4e0b-987d-9ec18b084229,20180731,1,900000000000207008,900000000000490003,3566051011,723277005
8,8000ba82-d025-41f8-89b4-22ea1ad7169e,20180731,1,900000000000207008,900000000000489007,429404000,723277005
9,8000ce76-3c37-50c1-a27f-def67af60e84,20020131,1,900000000000207008,900000000000489007,235772003,900000000000482003


## der2_Refset_SimpleFull_US1000124_20260301.txt

### Overview
The `der2_Refset_SimpleFull` file is a generic simple reference set. These refsets are used to identify a collection of concepts, descriptions, or relationships for a specific purpose, without adding any further qualifying information beyond the fact that they are members of the set.

### Column Descriptions
For simple refset files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the reference set itself.
*   **`referencedComponentId`**: Identifier of the component (concept, description, or relationship) that is a member of this refset.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Content', 'der2_Refset_SimpleFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_Refset_SimpleFull_US1000124_20260301.txt'

Successfully loaded 'der2_Refset_SimpleFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId
0,800aa109-431f-4407-a431-6fe65e9db160,20170731,1,900000000000207008,723264001,731819006
1,800aa109-431f-4407-a431-6fe65e9db160,20230228,0,900000000000207008,723264001,731819006
2,800c483a-ad68-4a07-a342-73ac73caa1b2,20170731,1,900000000000207008,723264001,728066008
3,800e34b5-2188-4345-b76e-389a3b1be1c1,20170131,1,900000000000207008,723264001,29376000
4,80114701-2c85-4a30-b176-5f9e1e34e47a,20170131,1,900000000000207008,723264001,87432006
5,8012555d-8ef7-4f99-a62e-bea638d196b4,20170731,1,900000000000207008,723264001,730908003
6,8016c259-37be-418b-b9c7-2147d615ea77,20170131,1,900000000000207008,723264001,180981003
7,801adf06-36ef-4f38-a0cf-d3da60d366e1,20170131,1,900000000000207008,723264001,150794004
8,801e48be-e2d6-4f86-bf07-c95919ba5b80,20170131,1,900000000000207008,723264001,705102002
9,801ec50f-5a46-44d7-9524-a1dd0971d52b,20170131,1,900000000000207008,723264001,714349005


### Full/Refset/Metadata

## der2_cciRefset_RefsetDescriptorFull_US1000124_20260301.txt

### Overview
The `der2_cciRefset_RefsetDescriptorFull` file provides metadata about reference sets themselves. It describes properties of refsets, such as their type, which components they can reference, and the attributes used within them. This file is crucial for understanding the structure and purpose of other refsets.

### Column Descriptions
For refset descriptor files, the columns typically include:
*   **`id`**: Unique identifier for the refset descriptor member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset descriptor is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the refset being described.
*   **`referencedComponentId`**: Identifier of the component that is typically referenced by this refset (e.g., a Concept ID).
*   **`attributeDescription`**: An identifier for a concept that describes the nature of the attribute within the refset.
*   **`attributeType`**: The data type of the attribute (e.g., string, integer, boolean).
*   **`attributeCardinality`**: The cardinality of the attribute (e.g., 0..1, 1..1).
*   **`attributeDataType`**: An identifier for the concept that represents the data type of the attribute.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_cciRefset_RefsetDescriptorFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_cciRefset_RefsetDescriptorFull_US1000124_20260301.txt'

Successfully loaded 'der2_cciRefset_RefsetDescriptorFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,attributeDescription,attributeType,attributeOrder
0,81ee191b-56fd-4950-9e14-6cb4c5cb5b1b,20170731,1,900000000000012004,900000000000456007,723561005,723574004,900000000000461009,6
1,8561568a-c56c-4b02-aaff-05e581606ebf,20140131,1,900000000000207008,900000000000456007,900000000000531004,900000000000533001,900000000000460005,1
2,85cc1c27-3a9f-5859-a1c5-7522b51ed528,20130731,1,449080006,900000000000456007,447562003,900000000000502003,900000000000478000,2
3,85d2cadc-65a0-5e0e-aa32-91236adfb1be,20020131,1,900000000000207008,900000000000456007,900000000000488004,449608002,900000000000463007,0
4,885e163a-5c99-423d-9879-dc9e10599fb5,20170731,1,900000000000207008,900000000000456007,900000000000496009,900000000000499002,900000000000465000,1
5,885e163a-5c99-423d-9879-dc9e10599fb5,20220131,0,900000000000207008,900000000000456007,900000000000496009,900000000000499002,900000000000465000,1
6,89613c1d-fff1-4744-951a-9c1ee341af5a,20170731,1,900000000000207008,900000000000456007,447258008,447255006,900000000000478000,1
7,89aa39de-d0a8-5879-9ea6-79a88e6d65de,20020131,1,900000000000207008,900000000000456007,900000000000526001,900000000000532006,900000000000460005,0
8,8af233ae-d139-4335-8bef-336862b68e89,20210131,1,900000000000012004,900000000000456007,1119435002,733616009,1119461003,2
9,8bc8366e-7cf1-55ef-9f90-872183466155,20020131,1,900000000000207008,900000000000456007,900000000000524003,900000000000533001,900000000000460005,1


## der2_ciRefset_DescriptionTypeFull_US1000124_20260301.txt

### Overview
The `der2_ciRefset_DescriptionTypeFull` file provides details about different types of descriptions used in SNOMED CT. It defines the properties and characteristics of description types, helping to categorize and manage the various ways concepts can be expressed textually.

### Column Descriptions
For description type refset files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the reference set itself.
*   **`referencedComponentId`**: Identifier of the concept representing the description type.
*   **`descriptionLength`**: The maximum length of the description string.
*   **`descriptionFormat`**: The format of the description (e.g., plain text, HTML).
*   **`characteristicTypeId`**: The concept ID for the characteristic type (e.g., case significance).
*   **`dataTypeId`**: The concept ID for the data type of the description type's attribute.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_ciRefset_DescriptionTypeFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_ciRefset_DescriptionTypeFull_US1000124_20260301.txt'

Successfully loaded 'der2_ciRefset_DescriptionTypeFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,descriptionFormat,descriptionLength
0,807f775b-1d66-5069-b58e-a37ace985dcf,20020131,1,900000000000207008,900000000000538005,900000000000550004,900000000000540000,1024
1,807f775b-1d66-5069-b58e-a37ace985dcf,20140131,1,900000000000207008,900000000000538005,900000000000550004,900000000000540000,4096
2,909a711e-b114-5543-841e-242aaa246363,20020131,1,900000000000207008,900000000000538005,900000000000013009,900000000000540000,255
3,0f928c01-b245-5907-9758-a46cbeed2674,20020131,1,900000000000207008,900000000000538005,900000000000003001,900000000000540000,255


### Snapshot/Terminology

## der2_cissccRefset_MRCMAttributeDomainFull_US1000124_20260301.txt

### Overview
The `der2_cissccRefset_MRCMAttributeDomainFull` file is part of the Machine Readable Concept Model (MRCM) and defines the domain of attributes. It specifies which concepts can have which attributes, and what values those attributes can take, ensuring consistency and validity in SNOMED CT content.

### Column Descriptions
For MRCM attribute domain files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the MRCM attribute domain reference set.
*   **`referencedComponentId`**: Identifier of the concept representing the attribute.
*   **`domainId`**: The identifier of the concept that serves as the domain for this attribute.
*   **`grouped`**: Boolean indicating if the attribute is grouped.
*   **`attributeCardinality`**: Specifies the allowed cardinality of the attribute (e.g., 0..1, 1..*).
*   **`attributeInGroupCardinality`**: Specifies the allowed cardinality of the attribute within a group.
*   **`ruleStrengthId`**: The strength of the rule (e.g., 'mandatory', 'optional').
*   **`contentTypeId`**: The type of content that the attribute applies to.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_cissccRefset_MRCMAttributeDomainFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_cissccRefset_MRCMAttributeDomainFull_US1000124_20260301.txt'

Successfully loaded 'der2_cissccRefset_MRCMAttributeDomainFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,domainId,grouped,attributeCardinality,attributeInGroupCardinality,ruleStrengthId,contentTypeId
0,8038688b-80f5-49b3-b3d8-c6fe13f8fb1e,20170731,1,900000000000012004,723561005,370129005,386053000,1,0..*,0..1,723597001,723596005
1,81da6aa5-0cb1-463f-88ad-c25a795a211b,20170731,1,900000000000012004,723561005,363713009,404684003,1,0..*,0..1,723597001,723596005
2,81eded9f-9769-e314-ca69-3b7e9e49c148,20250201,1,900000000000012004,723561005,116688005,71388002,1,0..*,0..1,723597001,723596005
3,840861c7-8d9b-44a6-acfd-01f1b994b6d9,20170731,1,900000000000012004,723561005,405813007,71388002,1,0..*,0..1,723597001,723596005
4,87e54647-dee0-4bd4-8b45-0ab0fa9fad74,20210731,1,900000000000012004,723561005,1148965004,260787004,0,0..1,0..0,723597001,723596005
5,883bd314-23b7-4ccc-87d6-8067d98b3660,20170731,1,900000000000012004,723561005,704324001,363787002,0,0..*,0..0,723597001,723596005
6,883bd314-23b7-4ccc-87d6-8067d98b3660,20180131,1,900000000000012004,723561005,704324001,363787002,1,0..*,0..1,723597001,723596005
7,89d0a100-7140-4913-bf80-bef3bc3d8495,20200731,1,900000000000012004,723561005,860779006,373873005,1,0..*,0..*,723597001,723596005
8,8ce25c12-3cc5-4ab0-84e2-fdc0a75dffed,20170731,1,900000000000012004,723561005,704327008,363787002,0,0..*,0..0,723597001,723596005
9,8ce25c12-3cc5-4ab0-84e2-fdc0a75dffed,20180131,1,900000000000012004,723561005,704327008,363787002,1,0..*,0..1,723597001,723596005


## der2_cRefset_MRCMModuleScopeFull_US1000124_20260301.txt

### Overview
The `der2_cRefset_MRCMModuleScopeFull` file defines the scope of the Machine Readable Concept Model (MRCM) rules for specific modules. It indicates which MRCM rules apply to which SNOMED CT modules, allowing for modular and scalable content modeling.

### Column Descriptions
For MRCM module scope files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the MRCM module scope reference set.
*   **`referencedComponentId`**: Identifier of the module to which the MRCM rules apply.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_cRefset_MRCMModuleScopeFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_cRefset_MRCMModuleScopeFull_US1000124_20260301.txt'

Successfully loaded 'der2_cRefset_MRCMModuleScopeFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,mrcmRuleRefsetId
0,8e5766bc-7755-45fb-99c8-8d2c52e45da5,20170731,1,900000000000012004,723563008,900000000000207008,723562003
1,d3b7a01f-6fcd-4d8d-815b-ca6a63511310,20170731,1,900000000000012004,723563008,900000000000207008,723561005
2,60dcf17f-6054-4ee2-a657-606104c49bb5,20170731,1,900000000000012004,723563008,900000000000207008,723560006


## der2_scsRefset_ComponentAnnotationStringValueFull_US1000124_20260301.txt

### Overview
The `der2_scsRefset_ComponentAnnotationStringValueFull` file provides string-based annotations for SNOMED CT components. These annotations can be used to attach arbitrary text values to concepts, descriptions, or relationships for various purposes, such as editorial notes or comments.

### Column Descriptions
For component annotation string value files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the annotation reference set.
*   **`referencedComponentId`**: Identifier of the SNOMED CT component being annotated.
*   **`stringAttributeValue`**: The string value of the annotation.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_scsRefset_ComponentAnnotationStringValueFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_scsRefset_ComponentAnnotationStringValueFull_US1000124_20260301.txt'

Successfully loaded 'der2_scsRefset_ComponentAnnotationStringValueFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,languageDialectCode,typeId,value
0,8000f421-3619-4607-8573-b7249677262c,20250201,1,900000000000207008,1292992004,770909004,en,1295448001,Inserm Orphanet
1,8017e26c-0999-40fa-a4ca-dfa88fe619f3,20250101,1,900000000000207008,1292992004,1222791008,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
2,8019e6c1-1612-4af1-b43e-3a49f5888214,20250101,1,900000000000207008,1292992004,1229852009,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
3,8028e13a-2b8c-4628-a588-607049b96468,20250101,1,900000000000207008,1292992004,1229851002,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
4,803dcbe9-3afc-4ef1-aebf-7ca4fbf432cc,20250101,1,900000000000207008,1292992004,719271000,en,1295448001,Inserm Orphanet
5,803ffc1d-4bfb-4d32-9a60-0c027cc858ae,20250201,1,900000000000207008,1292992004,721888002,en,1295448001,Inserm Orphanet
6,80431efd-ff00-476e-9656-b4f4aca971d2,20250201,1,900000000000207008,1292992004,770788000,en,1295448001,Inserm Orphanet
7,80446dc2-4094-454a-a7e5-f74ca5d943b8,20251001,1,900000000000207008,1292992004,1373339000,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
8,805366b6-854d-4003-900b-58f1835e9375,20250101,1,900000000000207008,1292992004,1229901006,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
9,805c83bf-747d-4a80-b4d6-217c35823a1f,20250201,1,900000000000207008,1292992004,725417001,en,1295448001,Inserm Orphanet


## der2_ssccRefset_MRCMAttributeRangeFull_US1000124_20260301.txt

### Overview
The `der2_ssccRefset_MRCMAttributeRangeFull` file specifies the valid range of values for attributes within the Machine Readable Concept Model (MRCM). This ensures that attribute values conform to predefined constraints, contributing to the quality and consistency of SNOMED CT content.

### Column Descriptions
For MRCM attribute range files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the MRCM attribute range reference set.
*   **`referencedComponentId`**: Identifier of the attribute for which the range is being defined.
*   **`rangeConstraint`**: The constraint defining the allowed range of values.
*   **`attributeRule`**: A rule that further specifies how the range applies.
*   **`ruleStrengthId`**: The strength of the rule.
*   **`contentTypeId`**: The type of content that the attribute applies to.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_ssccRefset_MRCMAttributeRangeFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_ssccRefset_MRCMAttributeRangeFull_US1000124_20260301.txt'

Successfully loaded 'der2_ssccRefset_MRCMAttributeRangeFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,rangeConstraint,attributeRule,ruleStrengthId,contentTypeId
0,81288567-57a9-49b1-b7f0-bf5979a6d289,20170731,1,900000000000012004,723562003,405814001,<< 442083009 |Anatomical or acquired body stru...,<< 71388002 |Procedure (procedure)|: [0..*] { ...,723597001,723596005
1,8163e3c0-b0c4-4c92-82e7-e93a23770b17,20170731,1,900000000000012004,723562003,246090004,<< 404684003 |Clinical finding (finding)| OR <...,<< 413350009 |Finding with explicit context (s...,723597001,723595009
2,8163e3c0-b0c4-4c92-82e7-e93a23770b17,20200731,1,900000000000012004,723562003,246090004,<< 272379006 |Event (event)| OR << 363787002 |...,<< 413350009 |Finding with explicit context (s...,723597001,723595009
3,8163e3c0-b0c4-4c92-82e7-e93a23770b17,20230630,1,900000000000012004,723562003,246090004,<< 272379006 |Event (event)| OR << 363787002 |...,<< 413350009 |Finding with explicit context (s...,723597001,723595009
4,81f6d23b-66c7-46ce-b003-08380b9b236a,20170731,1,900000000000012004,723562003,118170007,<< 125676002 |Person (person)| OR << 35359004 ...,<< 123038009 |Specimen (specimen)|: [0..*] { [...,723597001,723596005
5,81f6d23b-66c7-46ce-b003-08380b9b236a,20180731,1,900000000000012004,723562003,118170007,<< 125676002 |Person (person)| OR << 35359004 ...,<< 123038009 |Specimen (specimen)|: [0..*] { [...,723597001,723596005
6,81f6d23b-66c7-46ce-b003-08380b9b236a,20200731,1,900000000000012004,723562003,118170007,<< 125676002 |Person (person)| OR << 133928008...,<< 123038009 |Specimen (specimen)|: [0..*] { [...,723597001,723596005
7,8a4b2de3-41cf-49a4-995e-9f80c71e0684,20170731,1,900000000000012004,723562003,732943007,< 105590001 |Substance (substance)|,<< 373873005 |Pharmaceutical / biologic produc...,723597001,723596005
8,8a4b2de3-41cf-49a4-995e-9f80c71e0684,20200731,1,900000000000012004,723562003,732943007,< 105590001 |Substance (substance)|,<< 373873005 |Pharmaceutical / biologic produc...,723597001,723596005
9,903d4612-9e2e-4434-94e8-eaff214f4467,20170731,1,900000000000012004,723562003,425391005,<< 49062001 |Device (physical object)|,<< 71388002 |Procedure (procedure)|: [0..*] { ...,723597001,723596005


## der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt

### Overview
The `der2_sscsRefset_MemberAnnotationStringValueFull` file allows for attaching string-based annotations to reference set members. This is useful for providing additional descriptive information, comments, or provenance details about individual entries within a reference set.

### Column Descriptions
For member annotation string value files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the annotation reference set.
*   **`referencedComponentId`**: Identifier of the reference set member being annotated.
*   **`stringAttributeValue`**: The string value of the annotation.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt'

Successfully loaded 'der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,referencedMemberId,languageDialectCode,typeId,value


## der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt

### Overview
The `der2_ssRefset_ModuleDependencyFull` file specifies dependencies between SNOMED CT modules. It records which modules depend on other modules, which is essential for managing the modular release and extension of SNOMED CT content.

### Column Descriptions
For module dependency files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the module dependency reference set.
*   **`referencedComponentId`**: Identifier of the dependent module.
*   **`sourceEffectiveTime`**: The effective time of the source module.
*   **`targetEffectiveTime`**: The effective time of the target module.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt'

Successfully loaded 'der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,sourceEffectiveTime,targetEffectiveTime
0,9200c4da-3189-4dd5-9879-02c9ae118747,20220901,1,5991000124107,900000000000534007,900000000000207008,20220901,20220630
1,9200c4da-3189-4dd5-9879-02c9ae118747,20230301,1,5991000124107,900000000000534007,900000000000207008,20230301,20221231
2,9200c4da-3189-4dd5-9879-02c9ae118747,20230901,1,5991000124107,900000000000534007,900000000000207008,20230901,20230630
3,9200c4da-3189-4dd5-9879-02c9ae118747,20240301,1,5991000124107,900000000000534007,900000000000207008,20240301,20240101
4,9200c4da-3189-4dd5-9879-02c9ae118747,20240901,1,5991000124107,900000000000534007,900000000000207008,20240901,20240701
5,9200c4da-3189-4dd5-9879-02c9ae118747,20250301,1,5991000124107,900000000000534007,900000000000207008,20250301,20250101
6,9200c4da-3189-4dd5-9879-02c9ae118747,20250901,1,5991000124107,900000000000534007,900000000000207008,20250901,20250701
7,9200c4da-3189-4dd5-9879-02c9ae118747,20260301,1,5991000124107,900000000000534007,900000000000207008,20260301,20260101
8,ed482ae6-dff1-42d8-8e39-353b78a60471,20220901,1,5991000124107,900000000000534007,900000000000012004,20220901,20220630
9,ed482ae6-dff1-42d8-8e39-353b78a60471,20230301,1,5991000124107,900000000000534007,900000000000012004,20230301,20221231


## der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt

### Overview
The `der2_sssssssRefset_MRCMDomainFull` file defines the domain of concepts that are constrained by a particular MRCM rule. It specifies which subsets of SNOMED CT concepts are subject to certain modeling rules, ensuring that content development adheres to the established concept model.

### Column Descriptions
For MRCM domain files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the MRCM domain reference set.
*   **`referencedComponentId`**: Identifier of the concept representing the domain.
*   **`domainConstraint`**: The constraint that defines the domain.
*   **`parentDomain`**: The identifier of the parent domain, if applicable.
*   **`proximalPrimitiveConcept`**: The identifier of the closest primitive concept.
*   **`proximalPrimitiveRefinement`**: The refinement expression for the proximal primitive concept.
*   **`domainTemplateForPrecoordination`**: A template for precoordinated concepts.
*   **`domainTemplateForPostcoordination`**: A template for postcoordinated concepts.
*   **`guideURL`**: A URL to a guide or documentation related to the domain.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt'

Successfully loaded 'der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,domainConstraint,parentDomain,proximalPrimitiveConstraint,proximalPrimitiveRefinement,domainTemplateForPrecoordination,domainTemplateForPostcoordination,guideURL
0,821b5d55-2985-4293-98a9-2959a69b0b3b,20170731,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
1,821b5d55-2985-4293-98a9-2959a69b0b3b,20190731,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
2,821b5d55-2985-4293-98a9-2959a69b0b3b,20200731,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
3,821b5d55-2985-4293-98a9-2959a69b0b3b,20231001,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
4,821b5d55-2985-4293-98a9-2959a69b0b3b,20240501,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
5,821b5d55-2985-4293-98a9-2959a69b0b3b,20250201,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
6,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20170731,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [+id(<< 12926500...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000
7,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20180131,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [+id(<< 12926500...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000
8,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20190731,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [+id(<< 12926500...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000
9,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20200131,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [[+id(<< 1292650...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000


## der2_cRefset_MRCMModuleScopeFull_US1000124_20260301.txt

### Overview
The `der2_cRefset_MRCMModuleScopeFull` file defines the scope of the Machine Readable Concept Model (MRCM) rules for specific modules. It indicates which MRCM rules apply to which SNOMED CT modules, allowing for modular and scalable content modeling.

### Column Descriptions
For MRCM module scope files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the MRCM module scope reference set.
*   **`referencedComponentId`**: Identifier of the module to which the MRCM rules apply.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_cRefset_MRCMModuleScopeFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_cRefset_MRCMModuleScopeFull_US1000124_20260301.txt'

Successfully loaded 'der2_cRefset_MRCMModuleScopeFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,mrcmRuleRefsetId
0,8e5766bc-7755-45fb-99c8-8d2c52e45da5,20170731,1,900000000000012004,723563008,900000000000207008,723562003
1,d3b7a01f-6fcd-4d8d-815b-ca6a63511310,20170731,1,900000000000012004,723563008,900000000000207008,723561005
2,60dcf17f-6054-4ee2-a657-606104c49bb5,20170731,1,900000000000012004,723563008,900000000000207008,723560006


## der2_scsRefset_ComponentAnnotationStringValueFull_US1000124_20260301.txt

### Overview
The `der2_scsRefset_ComponentAnnotationStringValueFull` file provides string-based annotations for SNOMED CT components. These annotations can be used to attach arbitrary text values to concepts, descriptions, or relationships for various purposes, such as editorial notes or comments.

### Column Descriptions
For component annotation string value files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the annotation reference set.
*   **`referencedComponentId`**: Identifier of the SNOMED CT component being annotated.
*   **`stringAttributeValue`**: The string value of the annotation.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_scsRefset_ComponentAnnotationStringValueFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_scsRefset_ComponentAnnotationStringValueFull_US1000124_20260301.txt'

Successfully loaded 'der2_scsRefset_ComponentAnnotationStringValueFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,languageDialectCode,typeId,value
0,8000f421-3619-4607-8573-b7249677262c,20250201,1,900000000000207008,1292992004,770909004,en,1295448001,Inserm Orphanet
1,8017e26c-0999-40fa-a4ca-dfa88fe619f3,20250101,1,900000000000207008,1292992004,1222791008,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
2,8019e6c1-1612-4af1-b43e-3a49f5888214,20250101,1,900000000000207008,1292992004,1229852009,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
3,8028e13a-2b8c-4628-a588-607049b96468,20250101,1,900000000000207008,1292992004,1229851002,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
4,803dcbe9-3afc-4ef1-aebf-7ca4fbf432cc,20250101,1,900000000000207008,1292992004,719271000,en,1295448001,Inserm Orphanet
5,803ffc1d-4bfb-4d32-9a60-0c027cc858ae,20250201,1,900000000000207008,1292992004,721888002,en,1295448001,Inserm Orphanet
6,80431efd-ff00-476e-9656-b4f4aca971d2,20250201,1,900000000000207008,1292992004,770788000,en,1295448001,Inserm Orphanet
7,80446dc2-4094-454a-a7e5-f74ca5d943b8,20251001,1,900000000000207008,1292992004,1373339000,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
8,805366b6-854d-4003-900b-58f1835e9375,20250101,1,900000000000207008,1292992004,1229901006,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
9,805c83bf-747d-4a80-b4d6-217c35823a1f,20250201,1,900000000000207008,1292992004,725417001,en,1295448001,Inserm Orphanet


## der2_ssccRefset_MRCMAttributeRangeFull_US1000124_20260301.txt

### Overview
The `der2_ssccRefset_MRCMAttributeRangeFull` file specifies the valid range of values for attributes within the Machine Readable Concept Model (MRCM). This ensures that attribute values conform to predefined constraints, contributing to the quality and consistency of SNOMED CT content.

### Column Descriptions
For MRCM attribute range files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the MRCM attribute range reference set.
*   **`referencedComponentId`**: Identifier of the attribute for which the range is being defined.
*   **`rangeConstraint`**: The constraint defining the allowed range of values.
*   **`attributeRule`**: A rule that further specifies how the range applies.
*   **`ruleStrengthId`**: The strength of the rule.
*   **`contentTypeId`**: The type of content that the attribute applies to.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_ssccRefset_MRCMAttributeRangeFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_ssccRefset_MRCMAttributeRangeFull_US1000124_20260301.txt'

Successfully loaded 'der2_ssccRefset_MRCMAttributeRangeFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,rangeConstraint,attributeRule,ruleStrengthId,contentTypeId
0,81288567-57a9-49b1-b7f0-bf5979a6d289,20170731,1,900000000000012004,723562003,405814001,<< 442083009 |Anatomical or acquired body stru...,<< 71388002 |Procedure (procedure)|: [0..*] { ...,723597001,723596005
1,8163e3c0-b0c4-4c92-82e7-e93a23770b17,20170731,1,900000000000012004,723562003,246090004,<< 404684003 |Clinical finding (finding)| OR <...,<< 413350009 |Finding with explicit context (s...,723597001,723595009
2,8163e3c0-b0c4-4c92-82e7-e93a23770b17,20200731,1,900000000000012004,723562003,246090004,<< 272379006 |Event (event)| OR << 363787002 |...,<< 413350009 |Finding with explicit context (s...,723597001,723595009
3,8163e3c0-b0c4-4c92-82e7-e93a23770b17,20230630,1,900000000000012004,723562003,246090004,<< 272379006 |Event (event)| OR << 363787002 |...,<< 413350009 |Finding with explicit context (s...,723597001,723595009
4,81f6d23b-66c7-46ce-b003-08380b9b236a,20170731,1,900000000000012004,723562003,118170007,<< 125676002 |Person (person)| OR << 35359004 ...,<< 123038009 |Specimen (specimen)|: [0..*] { [...,723597001,723596005
5,81f6d23b-66c7-46ce-b003-08380b9b236a,20180731,1,900000000000012004,723562003,118170007,<< 125676002 |Person (person)| OR << 35359004 ...,<< 123038009 |Specimen (specimen)|: [0..*] { [...,723597001,723596005
6,81f6d23b-66c7-46ce-b003-08380b9b236a,20200731,1,900000000000012004,723562003,118170007,<< 125676002 |Person (person)| OR << 133928008...,<< 123038009 |Specimen (specimen)|: [0..*] { [...,723597001,723596005
7,8a4b2de3-41cf-49a4-995e-9f80c71e0684,20170731,1,900000000000012004,723562003,732943007,< 105590001 |Substance (substance)|,<< 373873005 |Pharmaceutical / biologic produc...,723597001,723596005
8,8a4b2de3-41cf-49a4-995e-9f80c71e0684,20200731,1,900000000000012004,723562003,732943007,< 105590001 |Substance (substance)|,<< 373873005 |Pharmaceutical / biologic produc...,723597001,723596005
9,903d4612-9e2e-4434-94e8-eaff214f4467,20170731,1,900000000000012004,723562003,425391005,<< 49062001 |Device (physical object)|,<< 71388002 |Procedure (procedure)|: [0..*] { ...,723597001,723596005


## der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt

### Overview
The `der2_sscsRefset_MemberAnnotationStringValueFull` file allows for attaching string-based annotations to reference set members. This is useful for providing additional descriptive information, comments, or provenance details about individual entries within a reference set.

### Column Descriptions
For member annotation string value files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the annotation reference set.
*   **`referencedComponentId`**: Identifier of the reference set member being annotated.
*   **`stringAttributeValue`**: The string value of the annotation.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt'

Successfully loaded 'der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,referencedMemberId,languageDialectCode,typeId,value


## der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt

### Overview
The `der2_ssRefset_ModuleDependencyFull` file specifies dependencies between SNOMED CT modules. It records which modules depend on other modules, which is essential for managing the modular release and extension of SNOMED CT content.

### Column Descriptions
For module dependency files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the module dependency reference set.
*   **`referencedComponentId`**: Identifier of the dependent module.
*   **`sourceEffectiveTime`**: The effective time of the source module.
*   **`targetEffectiveTime`**: The effective time of the target module.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt'

Successfully loaded 'der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,sourceEffectiveTime,targetEffectiveTime
0,9200c4da-3189-4dd5-9879-02c9ae118747,20220901,1,5991000124107,900000000000534007,900000000000207008,20220901,20220630
1,9200c4da-3189-4dd5-9879-02c9ae118747,20230301,1,5991000124107,900000000000534007,900000000000207008,20230301,20221231
2,9200c4da-3189-4dd5-9879-02c9ae118747,20230901,1,5991000124107,900000000000534007,900000000000207008,20230901,20230630
3,9200c4da-3189-4dd5-9879-02c9ae118747,20240301,1,5991000124107,900000000000534007,900000000000207008,20240301,20240101
4,9200c4da-3189-4dd5-9879-02c9ae118747,20240901,1,5991000124107,900000000000534007,900000000000207008,20240901,20240701
5,9200c4da-3189-4dd5-9879-02c9ae118747,20250301,1,5991000124107,900000000000534007,900000000000207008,20250301,20250101
6,9200c4da-3189-4dd5-9879-02c9ae118747,20250901,1,5991000124107,900000000000534007,900000000000207008,20250901,20250701
7,9200c4da-3189-4dd5-9879-02c9ae118747,20260301,1,5991000124107,900000000000534007,900000000000207008,20260301,20260101
8,ed482ae6-dff1-42d8-8e39-353b78a60471,20220901,1,5991000124107,900000000000534007,900000000000012004,20220901,20220630
9,ed482ae6-dff1-42d8-8e39-353b78a60471,20230301,1,5991000124107,900000000000534007,900000000000012004,20230301,20221231


## der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt

### Overview
The `der2_sssssssRefset_MRCMDomainFull` file defines the domain of concepts that are constrained by a particular MRCM rule. It specifies which subsets of SNOMED CT concepts are subject to certain modeling rules, ensuring that content development adheres to the established concept model.

### Column Descriptions
For MRCM domain files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the MRCM domain reference set.
*   **`referencedComponentId`**: Identifier of the concept representing the domain.
*   **`domainConstraint`**: The constraint that defines the domain.
*   **`parentDomain`**: The identifier of the parent domain, if applicable.
*   **`proximalPrimitiveConcept`**: The identifier of the closest primitive concept.
*   **`proximalPrimitiveRefinement`**: The refinement expression for the proximal primitive concept.
*   **`domainTemplateForPrecoordination`**: A template for precoordinated concepts.
*   **`domainTemplateForPostcoordination`**: A template for postcoordinated concepts.
*   **`guideURL`**: A URL to a guide or documentation related to the domain.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt'

Successfully loaded 'der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,domainConstraint,parentDomain,proximalPrimitiveConstraint,proximalPrimitiveRefinement,domainTemplateForPrecoordination,domainTemplateForPostcoordination,guideURL
0,821b5d55-2985-4293-98a9-2959a69b0b3b,20170731,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
1,821b5d55-2985-4293-98a9-2959a69b0b3b,20190731,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
2,821b5d55-2985-4293-98a9-2959a69b0b3b,20200731,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
3,821b5d55-2985-4293-98a9-2959a69b0b3b,20231001,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
4,821b5d55-2985-4293-98a9-2959a69b0b3b,20240501,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
5,821b5d55-2985-4293-98a9-2959a69b0b3b,20250201,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
6,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20170731,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [+id(<< 12926500...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000
7,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20180131,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [+id(<< 12926500...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000
8,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20190731,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [+id(<< 12926500...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000
9,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20200131,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [[+id(<< 1292650...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000


## der2_scsRefset_ComponentAnnotationStringValueFull_US1000124_20260301.txt

### Overview
The `der2_scsRefset_ComponentAnnotationStringValueFull` file provides string-based annotations for SNOMED CT components. These annotations can be used to attach arbitrary text values to concepts, descriptions, or relationships for various purposes, such as editorial notes or comments.

### Column Descriptions
For component annotation string value files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the annotation reference set.
*   **`referencedComponentId`**: Identifier of the SNOMED CT component being annotated.
*   **`stringAttributeValue`**: The string value of the annotation.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_scsRefset_ComponentAnnotationStringValueFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_scsRefset_ComponentAnnotationStringValueFull_US1000124_20260301.txt'

Successfully loaded 'der2_scsRefset_ComponentAnnotationStringValueFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,languageDialectCode,typeId,value
0,8000f421-3619-4607-8573-b7249677262c,20250201,1,900000000000207008,1292992004,770909004,en,1295448001,Inserm Orphanet
1,8017e26c-0999-40fa-a4ca-dfa88fe619f3,20250101,1,900000000000207008,1292992004,1222791008,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
2,8019e6c1-1612-4af1-b43e-3a49f5888214,20250101,1,900000000000207008,1292992004,1229852009,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
3,8028e13a-2b8c-4628-a588-607049b96468,20250101,1,900000000000207008,1292992004,1229851002,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
4,803dcbe9-3afc-4ef1-aebf-7ca4fbf432cc,20250101,1,900000000000207008,1292992004,719271000,en,1295448001,Inserm Orphanet
5,803ffc1d-4bfb-4d32-9a60-0c027cc858ae,20250201,1,900000000000207008,1292992004,721888002,en,1295448001,Inserm Orphanet
6,80431efd-ff00-476e-9656-b4f4aca971d2,20250201,1,900000000000207008,1292992004,770788000,en,1295448001,Inserm Orphanet
7,80446dc2-4094-454a-a7e5-f74ca5d943b8,20251001,1,900000000000207008,1292992004,1373339000,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
8,805366b6-854d-4003-900b-58f1835e9375,20250101,1,900000000000207008,1292992004,1229901006,en,1295448001,"American College of Surgeons, Chicago, Illinoi..."
9,805c83bf-747d-4a80-b4d6-217c35823a1f,20250201,1,900000000000207008,1292992004,725417001,en,1295448001,Inserm Orphanet


## der2_ssccRefset_MRCMAttributeRangeFull_US1000124_20260301.txt

### Overview
The `der2_ssccRefset_MRCMAttributeRangeFull` file specifies the valid range of values for attributes within the Machine Readable Concept Model (MRCM). This ensures that attribute values conform to predefined constraints, contributing to the quality and consistency of SNOMED CT content.

### Column Descriptions
For MRCM attribute range files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the MRCM attribute range reference set.
*   **`referencedComponentId`**: Identifier of the attribute for which the range is being defined.
*   **`rangeConstraint`**: The constraint defining the allowed range of values.
*   **`attributeRule`**: A rule that further specifies how the range applies.
*   **`ruleStrengthId`**: The strength of the rule.
*   **`contentTypeId`**: The type of content that the attribute applies to.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_ssccRefset_MRCMAttributeRangeFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_ssccRefset_MRCMAttributeRangeFull_US1000124_20260301.txt'

Successfully loaded 'der2_ssccRefset_MRCMAttributeRangeFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,rangeConstraint,attributeRule,ruleStrengthId,contentTypeId
0,81288567-57a9-49b1-b7f0-bf5979a6d289,20170731,1,900000000000012004,723562003,405814001,<< 442083009 |Anatomical or acquired body stru...,<< 71388002 |Procedure (procedure)|: [0..*] { ...,723597001,723596005
1,8163e3c0-b0c4-4c92-82e7-e93a23770b17,20170731,1,900000000000012004,723562003,246090004,<< 404684003 |Clinical finding (finding)| OR <...,<< 413350009 |Finding with explicit context (s...,723597001,723595009
2,8163e3c0-b0c4-4c92-82e7-e93a23770b17,20200731,1,900000000000012004,723562003,246090004,<< 272379006 |Event (event)| OR << 363787002 |...,<< 413350009 |Finding with explicit context (s...,723597001,723595009
3,8163e3c0-b0c4-4c92-82e7-e93a23770b17,20230630,1,900000000000012004,723562003,246090004,<< 272379006 |Event (event)| OR << 363787002 |...,<< 413350009 |Finding with explicit context (s...,723597001,723595009
4,81f6d23b-66c7-46ce-b003-08380b9b236a,20170731,1,900000000000012004,723562003,118170007,<< 125676002 |Person (person)| OR << 35359004 ...,<< 123038009 |Specimen (specimen)|: [0..*] { [...,723597001,723596005
5,81f6d23b-66c7-46ce-b003-08380b9b236a,20180731,1,900000000000012004,723562003,118170007,<< 125676002 |Person (person)| OR << 35359004 ...,<< 123038009 |Specimen (specimen)|: [0..*] { [...,723597001,723596005
6,81f6d23b-66c7-46ce-b003-08380b9b236a,20200731,1,900000000000012004,723562003,118170007,<< 125676002 |Person (person)| OR << 133928008...,<< 123038009 |Specimen (specimen)|: [0..*] { [...,723597001,723596005
7,8a4b2de3-41cf-49a4-995e-9f80c71e0684,20170731,1,900000000000012004,723562003,732943007,< 105590001 |Substance (substance)|,<< 373873005 |Pharmaceutical / biologic produc...,723597001,723596005
8,8a4b2de3-41cf-49a4-995e-9f80c71e0684,20200731,1,900000000000012004,723562003,732943007,< 105590001 |Substance (substance)|,<< 373873005 |Pharmaceutical / biologic produc...,723597001,723596005
9,903d4612-9e2e-4434-94e8-eaff214f4467,20170731,1,900000000000012004,723562003,425391005,<< 49062001 |Device (physical object)|,<< 71388002 |Procedure (procedure)|: [0..*] { ...,723597001,723596005


## der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt

### Overview
The `der2_sscsRefset_MemberAnnotationStringValueFull` file allows for attaching string-based annotations to reference set members. This is useful for providing additional descriptive information, comments, or provenance details about individual entries within a reference set.

### Column Descriptions
For member annotation string value files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the annotation reference set.
*   **`referencedComponentId`**: Identifier of the reference set member being annotated.
*   **`stringAttributeValue`**: The string value of the annotation.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt'

Successfully loaded 'der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,referencedMemberId,languageDialectCode,typeId,value


## der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt

### Overview
The `der2_ssRefset_ModuleDependencyFull` file specifies dependencies between SNOMED CT modules. It records which modules depend on other modules, which is essential for managing the modular release and extension of SNOMED CT content.

### Column Descriptions
For module dependency files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the module dependency reference set.
*   **`referencedComponentId`**: Identifier of the dependent module.
*   **`sourceEffectiveTime`**: The effective time of the source module.
*   **`targetEffectiveTime`**: The effective time of the target module.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt'

Successfully loaded 'der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,sourceEffectiveTime,targetEffectiveTime
0,9200c4da-3189-4dd5-9879-02c9ae118747,20220901,1,5991000124107,900000000000534007,900000000000207008,20220901,20220630
1,9200c4da-3189-4dd5-9879-02c9ae118747,20230301,1,5991000124107,900000000000534007,900000000000207008,20230301,20221231
2,9200c4da-3189-4dd5-9879-02c9ae118747,20230901,1,5991000124107,900000000000534007,900000000000207008,20230901,20230630
3,9200c4da-3189-4dd5-9879-02c9ae118747,20240301,1,5991000124107,900000000000534007,900000000000207008,20240301,20240101
4,9200c4da-3189-4dd5-9879-02c9ae118747,20240901,1,5991000124107,900000000000534007,900000000000207008,20240901,20240701
5,9200c4da-3189-4dd5-9879-02c9ae118747,20250301,1,5991000124107,900000000000534007,900000000000207008,20250301,20250101
6,9200c4da-3189-4dd5-9879-02c9ae118747,20250901,1,5991000124107,900000000000534007,900000000000207008,20250901,20250701
7,9200c4da-3189-4dd5-9879-02c9ae118747,20260301,1,5991000124107,900000000000534007,900000000000207008,20260301,20260101
8,ed482ae6-dff1-42d8-8e39-353b78a60471,20220901,1,5991000124107,900000000000534007,900000000000012004,20220901,20220630
9,ed482ae6-dff1-42d8-8e39-353b78a60471,20230301,1,5991000124107,900000000000534007,900000000000012004,20230301,20221231


## der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt

### Overview
The `der2_sssssssRefset_MRCMDomainFull` file defines the domain of concepts that are constrained by a particular MRCM rule. It specifies which subsets of SNOMED CT concepts are subject to certain modeling rules, ensuring that content development adheres to the established concept model.

### Column Descriptions
For MRCM domain files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the MRCM domain reference set.
*   **`referencedComponentId`**: Identifier of the concept representing the domain.
*   **`domainConstraint`**: The constraint that defines the domain.
*   **`parentDomain`**: The identifier of the parent domain, if applicable.
*   **`proximalPrimitiveConcept`**: The identifier of the closest primitive concept.
*   **`proximalPrimitiveRefinement`**: The refinement expression for the proximal primitive concept.
*   **`domainTemplateForPrecoordination`**: A template for precoordinated concepts.
*   **`domainTemplateForPostcoordination`**: A template for postcoordinated concepts.
*   **`guideURL`**: A URL to a guide or documentation related to the domain.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt'

Successfully loaded 'der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,domainConstraint,parentDomain,proximalPrimitiveConstraint,proximalPrimitiveRefinement,domainTemplateForPrecoordination,domainTemplateForPostcoordination,guideURL
0,821b5d55-2985-4293-98a9-2959a69b0b3b,20170731,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
1,821b5d55-2985-4293-98a9-2959a69b0b3b,20190731,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
2,821b5d55-2985-4293-98a9-2959a69b0b3b,20200731,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
3,821b5d55-2985-4293-98a9-2959a69b0b3b,20231001,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
4,821b5d55-2985-4293-98a9-2959a69b0b3b,20240501,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
5,821b5d55-2985-4293-98a9-2959a69b0b3b,20250201,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
6,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20170731,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [+id(<< 12926500...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000
7,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20180131,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [+id(<< 12926500...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000
8,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20190731,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [+id(<< 12926500...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000
9,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20200131,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [[+id(<< 1292650...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000


## der2_ssccRefset_MRCMAttributeRangeFull_US1000124_20260301.txt

### Overview
The `der2_ssccRefset_MRCMAttributeRangeFull` file specifies the valid range of values for attributes within the Machine Readable Concept Model (MRCM). This ensures that attribute values conform to predefined constraints, contributing to the quality and consistency of SNOMED CT content.

### Column Descriptions
For MRCM attribute range files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the MRCM attribute range reference set.
*   **`referencedComponentId`**: Identifier of the attribute for which the range is being defined.
*   **`rangeConstraint`**: The constraint defining the allowed range of values.
*   **`attributeRule`**: A rule that further specifies how the range applies.
*   **`ruleStrengthId`**: The strength of the rule.
*   **`contentTypeId`**: The type of content that the attribute applies to.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_ssccRefset_MRCMAttributeRangeFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_ssccRefset_MRCMAttributeRangeFull_US1000124_20260301.txt'

Successfully loaded 'der2_ssccRefset_MRCMAttributeRangeFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,rangeConstraint,attributeRule,ruleStrengthId,contentTypeId
0,81288567-57a9-49b1-b7f0-bf5979a6d289,20170731,1,900000000000012004,723562003,405814001,<< 442083009 |Anatomical or acquired body stru...,<< 71388002 |Procedure (procedure)|: [0..*] { ...,723597001,723596005
1,8163e3c0-b0c4-4c92-82e7-e93a23770b17,20170731,1,900000000000012004,723562003,246090004,<< 404684003 |Clinical finding (finding)| OR <...,<< 413350009 |Finding with explicit context (s...,723597001,723595009
2,8163e3c0-b0c4-4c92-82e7-e93a23770b17,20200731,1,900000000000012004,723562003,246090004,<< 272379006 |Event (event)| OR << 363787002 |...,<< 413350009 |Finding with explicit context (s...,723597001,723595009
3,8163e3c0-b0c4-4c92-82e7-e93a23770b17,20230630,1,900000000000012004,723562003,246090004,<< 272379006 |Event (event)| OR << 363787002 |...,<< 413350009 |Finding with explicit context (s...,723597001,723595009
4,81f6d23b-66c7-46ce-b003-08380b9b236a,20170731,1,900000000000012004,723562003,118170007,<< 125676002 |Person (person)| OR << 35359004 ...,<< 123038009 |Specimen (specimen)|: [0..*] { [...,723597001,723596005
5,81f6d23b-66c7-46ce-b003-08380b9b236a,20180731,1,900000000000012004,723562003,118170007,<< 125676002 |Person (person)| OR << 35359004 ...,<< 123038009 |Specimen (specimen)|: [0..*] { [...,723597001,723596005
6,81f6d23b-66c7-46ce-b003-08380b9b236a,20200731,1,900000000000012004,723562003,118170007,<< 125676002 |Person (person)| OR << 133928008...,<< 123038009 |Specimen (specimen)|: [0..*] { [...,723597001,723596005
7,8a4b2de3-41cf-49a4-995e-9f80c71e0684,20170731,1,900000000000012004,723562003,732943007,< 105590001 |Substance (substance)|,<< 373873005 |Pharmaceutical / biologic produc...,723597001,723596005
8,8a4b2de3-41cf-49a4-995e-9f80c71e0684,20200731,1,900000000000012004,723562003,732943007,< 105590001 |Substance (substance)|,<< 373873005 |Pharmaceutical / biologic produc...,723597001,723596005
9,903d4612-9e2e-4434-94e8-eaff214f4467,20170731,1,900000000000012004,723562003,425391005,<< 49062001 |Device (physical object)|,<< 71388002 |Procedure (procedure)|: [0..*] { ...,723597001,723596005


## der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt

### Overview
The `der2_sscsRefset_MemberAnnotationStringValueFull` file allows for attaching string-based annotations to reference set members. This is useful for providing additional descriptive information, comments, or provenance details about individual entries within a reference set.

### Column Descriptions
For member annotation string value files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the annotation reference set.
*   **`referencedComponentId`**: Identifier of the reference set member being annotated.
*   **`stringAttributeValue`**: The string value of the annotation.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt'

Successfully loaded 'der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,referencedMemberId,languageDialectCode,typeId,value


## der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt

### Overview
The `der2_ssRefset_ModuleDependencyFull` file specifies dependencies between SNOMED CT modules. It records which modules depend on other modules, which is essential for managing the modular release and extension of SNOMED CT content.

### Column Descriptions
For module dependency files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the module dependency reference set.
*   **`referencedComponentId`**: Identifier of the dependent module.
*   **`sourceEffectiveTime`**: The effective time of the source module.
*   **`targetEffectiveTime`**: The effective time of the target module.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt'

Successfully loaded 'der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,sourceEffectiveTime,targetEffectiveTime
0,9200c4da-3189-4dd5-9879-02c9ae118747,20220901,1,5991000124107,900000000000534007,900000000000207008,20220901,20220630
1,9200c4da-3189-4dd5-9879-02c9ae118747,20230301,1,5991000124107,900000000000534007,900000000000207008,20230301,20221231
2,9200c4da-3189-4dd5-9879-02c9ae118747,20230901,1,5991000124107,900000000000534007,900000000000207008,20230901,20230630
3,9200c4da-3189-4dd5-9879-02c9ae118747,20240301,1,5991000124107,900000000000534007,900000000000207008,20240301,20240101
4,9200c4da-3189-4dd5-9879-02c9ae118747,20240901,1,5991000124107,900000000000534007,900000000000207008,20240901,20240701
5,9200c4da-3189-4dd5-9879-02c9ae118747,20250301,1,5991000124107,900000000000534007,900000000000207008,20250301,20250101
6,9200c4da-3189-4dd5-9879-02c9ae118747,20250901,1,5991000124107,900000000000534007,900000000000207008,20250901,20250701
7,9200c4da-3189-4dd5-9879-02c9ae118747,20260301,1,5991000124107,900000000000534007,900000000000207008,20260301,20260101
8,ed482ae6-dff1-42d8-8e39-353b78a60471,20220901,1,5991000124107,900000000000534007,900000000000012004,20220901,20220630
9,ed482ae6-dff1-42d8-8e39-353b78a60471,20230301,1,5991000124107,900000000000534007,900000000000012004,20230301,20221231


## der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt

### Overview
The `der2_sssssssRefset_MRCMDomainFull` file defines the domain of concepts that are constrained by a particular MRCM rule. It specifies which subsets of SNOMED CT concepts are subject to certain modeling rules, ensuring that content development adheres to the established concept model.

### Column Descriptions
For MRCM domain files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the MRCM domain reference set.
*   **`referencedComponentId`**: Identifier of the concept representing the domain.
*   **`domainConstraint`**: The constraint that defines the domain.
*   **`parentDomain`**: The identifier of the parent domain, if applicable.
*   **`proximalPrimitiveConcept`**: The identifier of the closest primitive concept.
*   **`proximalPrimitiveRefinement`**: The refinement expression for the proximal primitive concept.
*   **`domainTemplateForPrecoordination`**: A template for precoordinated concepts.
*   **`domainTemplateForPostcoordination`**: A template for postcoordinated concepts.
*   **`guideURL`**: A URL to a guide or documentation related to the domain.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt'

Successfully loaded 'der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,domainConstraint,parentDomain,proximalPrimitiveConstraint,proximalPrimitiveRefinement,domainTemplateForPrecoordination,domainTemplateForPostcoordination,guideURL
0,821b5d55-2985-4293-98a9-2959a69b0b3b,20170731,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
1,821b5d55-2985-4293-98a9-2959a69b0b3b,20190731,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
2,821b5d55-2985-4293-98a9-2959a69b0b3b,20200731,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
3,821b5d55-2985-4293-98a9-2959a69b0b3b,20231001,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
4,821b5d55-2985-4293-98a9-2959a69b0b3b,20240501,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
5,821b5d55-2985-4293-98a9-2959a69b0b3b,20250201,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
6,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20170731,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [+id(<< 12926500...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000
7,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20180131,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [+id(<< 12926500...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000
8,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20190731,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [+id(<< 12926500...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000
9,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20200131,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [[+id(<< 1292650...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000


## der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt

### Overview
The `der2_sscsRefset_MemberAnnotationStringValueFull` file allows for attaching string-based annotations to reference set members. This is useful for providing additional descriptive information, comments, or provenance details about individual entries within a reference set.

### Column Descriptions
For member annotation string value files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the annotation reference set.
*   **`referencedComponentId`**: Identifier of the reference set member being annotated.
*   **`stringAttributeValue`**: The string value of the annotation.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt'

Successfully loaded 'der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,referencedMemberId,languageDialectCode,typeId,value


## der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt

### Overview
The `der2_ssRefset_ModuleDependencyFull` file specifies dependencies between SNOMED CT modules. It records which modules depend on other modules, which is essential for managing the modular release and extension of SNOMED CT content.

### Column Descriptions
For module dependency files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the module dependency reference set.
*   **`referencedComponentId`**: Identifier of the dependent module.
*   **`sourceEffectiveTime`**: The effective time of the source module.
*   **`targetEffectiveTime`**: The effective time of the target module.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt'

Successfully loaded 'der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,sourceEffectiveTime,targetEffectiveTime
0,9200c4da-3189-4dd5-9879-02c9ae118747,20220901,1,5991000124107,900000000000534007,900000000000207008,20220901,20220630
1,9200c4da-3189-4dd5-9879-02c9ae118747,20230301,1,5991000124107,900000000000534007,900000000000207008,20230301,20221231
2,9200c4da-3189-4dd5-9879-02c9ae118747,20230901,1,5991000124107,900000000000534007,900000000000207008,20230901,20230630
3,9200c4da-3189-4dd5-9879-02c9ae118747,20240301,1,5991000124107,900000000000534007,900000000000207008,20240301,20240101
4,9200c4da-3189-4dd5-9879-02c9ae118747,20240901,1,5991000124107,900000000000534007,900000000000207008,20240901,20240701
5,9200c4da-3189-4dd5-9879-02c9ae118747,20250301,1,5991000124107,900000000000534007,900000000000207008,20250301,20250101
6,9200c4da-3189-4dd5-9879-02c9ae118747,20250901,1,5991000124107,900000000000534007,900000000000207008,20250901,20250701
7,9200c4da-3189-4dd5-9879-02c9ae118747,20260301,1,5991000124107,900000000000534007,900000000000207008,20260301,20260101
8,ed482ae6-dff1-42d8-8e39-353b78a60471,20220901,1,5991000124107,900000000000534007,900000000000012004,20220901,20220630
9,ed482ae6-dff1-42d8-8e39-353b78a60471,20230301,1,5991000124107,900000000000534007,900000000000012004,20230301,20221231


## der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt

### Overview
The `der2_sssssssRefset_MRCMDomainFull` file defines the domain of concepts that are constrained by a particular MRCM rule. It specifies which subsets of SNOMED CT concepts are subject to certain modeling rules, ensuring that content development adheres to the established concept model.

### Column Descriptions
For MRCM domain files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the MRCM domain reference set.
*   **`referencedComponentId`**: Identifier of the concept representing the domain.
*   **`domainConstraint`**: The constraint that defines the domain.
*   **`parentDomain`**: The identifier of the parent domain, if applicable.
*   **`proximalPrimitiveConcept`**: The identifier of the closest primitive concept.
*   **`proximalPrimitiveRefinement`**: The refinement expression for the proximal primitive concept.
*   **`domainTemplateForPrecoordination`**: A template for precoordinated concepts.
*   **`domainTemplateForPostcoordination`**: A template for postcoordinated concepts.
*   **`guideURL`**: A URL to a guide or documentation related to the domain.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt'

Successfully loaded 'der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,domainConstraint,parentDomain,proximalPrimitiveConstraint,proximalPrimitiveRefinement,domainTemplateForPrecoordination,domainTemplateForPostcoordination,guideURL
0,821b5d55-2985-4293-98a9-2959a69b0b3b,20170731,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
1,821b5d55-2985-4293-98a9-2959a69b0b3b,20190731,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
2,821b5d55-2985-4293-98a9-2959a69b0b3b,20200731,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
3,821b5d55-2985-4293-98a9-2959a69b0b3b,20231001,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
4,821b5d55-2985-4293-98a9-2959a69b0b3b,20240501,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
5,821b5d55-2985-4293-98a9-2959a69b0b3b,20250201,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
6,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20170731,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [+id(<< 12926500...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000
7,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20180131,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [+id(<< 12926500...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000
8,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20190731,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [+id(<< 12926500...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000
9,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20200131,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [[+id(<< 1292650...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000


## der2_cRefset_AssociationSnapshot_US1000124_20260301.txt

### Overview
The `der2_cRefset_AssociationSnapshot` file is the snapshot version of the Association Reference Set. It contains the most current associations between concepts or between concepts and values (e.g., numerical values) within SNOMED CT. This file provides the latest state of these specific, detailed characteristics of SNOMED CT concepts, without historical versions.

### Column Descriptions
For association refset snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the reference set itself.
*   **`referencedComponentId`**: Identifier of the SNOMED CT concept to which the association applies.
*   **`targetComponentId`**: Identifier of the target concept or value in the association.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Refset', 'Content', 'der2_cRefset_AssociationSnapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_cRefset_AssociationSnapshot_US1000124_20260301.txt'

Successfully loaded 'der2_cRefset_AssociationSnapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,targetComponentId
0,80001d7e-b1b9-56ac-9768-308cabe31117,20040731,1,900000000000207008,900000000000527005,290170004,216464004
1,80006a57-2ab6-524d-8cb5-ab051fb10aaa,20020131,1,900000000000207008,900000000000527005,165811000,313476009
2,8001aeda-719f-5d07-a4aa-00b734b748da,20020731,1,900000000000207008,900000000000523009,266866008,159972006
3,8001b97b-6cb7-41e4-b3d4-eb823a0559a6,20230531,1,900000000000207008,900000000000523009,246531008,128612007
4,80024464-d5ea-4a12-b933-606a6056c06f,20170731,1,900000000000207008,734138000,280164007,729256003
5,80029c0f-4be7-40c6-ac40-f707251b1f07,20241001,1,900000000000207008,900000000000526001,404148006,109969005
6,8002dce0-0f88-48f4-af67-5e8626a427fb,20210131,1,900000000000207008,900000000000523009,119310001,1003709001
7,80030572-b3de-5f0c-a1c6-86a31ca0b807,20090131,1,900000000000207008,900000000000524003,316645000,370137002
8,80033ca4-4877-4b13-b6fe-11f1604685de,20190131,1,900000000000207008,900000000000523009,14255005,46799006
9,80033dfe-45a5-579c-b6da-9c5035fca20c,20020131,1,900000000000207008,900000000000527005,138488004,315509001


## der2_cRefset_AttributeValueSnapshot_US1000124_20260301.txt

### Overview
The `der2_cRefset_AttributeValueSnapshot` file is the snapshot version of the Attribute Value Reference Set. It contains the most current attribute values for specific SNOMED CT concepts. This file provides the latest descriptive information about concepts, such as clinical findings, procedures, or substances, without historical versions.

### Column Descriptions
For attribute value refset snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the reference set itself.
*   **`referencedComponentId`**: Identifier of the SNOMED CT concept to which the attribute value applies.
*   **`valueId`**: Identifier of the concept representing the attribute's value. In some cases, this could be a direct numerical or string value rather than a concept ID, depending on the attribute type.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Refset', 'Content', 'der2_cRefset_AttributeValueSnapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_cRefset_AttributeValueSnapshot_US1000124_20260301.txt'

Successfully loaded 'der2_cRefset_AttributeValueSnapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,valueId
0,80005357-6c6d-4f02-9532-67c73e3eccc7,20190131,1,900000000000207008,900000000000490003,3672851010,723277005
1,80005fab-e950-433c-88ef-1575d3cf4de5,20221130,1,900000000000207008,900000000000490003,391438018,723277005
2,80006347-0187-5de0-9c84-ea1ddf9e3986,20020131,1,900000000000207008,900000000000489007,193284003,900000000000482003
3,800089cc-3dd8-4840-a613-99e1b3a4b6ee,20180731,1,900000000000207008,900000000000490003,3567548019,723277005
4,80009975-8e9d-5238-8e5b-146db8b24013,20080731,0,900000000000207008,900000000000490003,777428015,900000000000495008
5,80009cd8-8d1d-414b-99e3-48d909b07812,20180731,1,900000000000207008,900000000000490003,2835304018,723277005
6,8000ba3d-f802-4e0b-987d-9ec18b084229,20180731,1,900000000000207008,900000000000490003,3566051011,723277005
7,8000ba82-d025-41f8-89b4-22ea1ad7169e,20180731,1,900000000000207008,900000000000489007,429404000,723277005
8,8000ce76-3c37-50c1-a27f-def67af60e84,20030731,1,900000000000207008,900000000000489007,235772003,900000000000484002
9,8000d9a0-5371-43bd-9ade-5740f0f3a0ea,20260101,0,900000000000207008,900000000000490003,308814019,900000000000495008


## der2_Refset_SimpleSnapshot_US1000124_20260301.txt

### Overview
The `der2_Refset_SimpleSnapshot` file is the snapshot version of a generic Simple Reference Set. These refsets are used to identify a collection of concepts, descriptions, or relationships for a specific purpose, without adding any further qualifying information beyond the fact that they are members of the set. This snapshot file provides the latest active state of such simple sets.

### Column Descriptions
For simple refset snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the reference set itself.
*   **`referencedComponentId`**: Identifier of the component (concept, description, or relationship) that is a member of this refset.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Refset', 'Content', 'der2_Refset_SimpleSnapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_Refset_SimpleSnapshot_US1000124_20260301.txt'

Successfully loaded 'der2_Refset_SimpleSnapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId
0,800aa109-431f-4407-a431-6fe65e9db160,20230228,0,900000000000207008,723264001,731819006
1,800c483a-ad68-4a07-a342-73ac73caa1b2,20170731,1,900000000000207008,723264001,728066008
2,800e34b5-2188-4345-b76e-389a3b1be1c1,20170131,1,900000000000207008,723264001,29376000
3,80114701-2c85-4a30-b176-5f9e1e34e47a,20170131,1,900000000000207008,723264001,87432006
4,8012555d-8ef7-4f99-a62e-bea638d196b4,20170731,1,900000000000207008,723264001,730908003
5,8016c259-37be-418b-b9c7-2147d615ea77,20170131,1,900000000000207008,723264001,180981003
6,801adf06-36ef-4f38-a0cf-d3da60d366e1,20170131,1,900000000000207008,723264001,150794004
7,801e48be-e2d6-4f86-bf07-c95919ba5b80,20170131,1,900000000000207008,723264001,705102002
8,801ec50f-5a46-44d7-9524-a1dd0971d52b,20170131,1,900000000000207008,723264001,714349005
9,801ee092-89e5-4628-9573-0a139b3422ea,20170131,1,900000000000207008,723264001,42981008


### Snapshot/Refset/Metadata

## der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt

### Overview
The `der2_sscsRefset_MemberAnnotationStringValueFull` file allows for attaching string-based annotations to reference set members. This is useful for providing additional descriptive information, comments, or provenance details about individual entries within a reference set.

### Column Descriptions
For member annotation string value files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the annotation reference set.
*   **`referencedComponentId`**: Identifier of the reference set member being annotated.
*   **`stringAttributeValue`**: The string value of the annotation.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt'

Successfully loaded 'der2_sscsRefset_MemberAnnotationStringValueFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,referencedMemberId,languageDialectCode,typeId,value


## der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt

### Overview
The `der2_ssRefset_ModuleDependencyFull` file specifies dependencies between SNOMED CT modules. It records which modules depend on other modules, which is essential for managing the modular release and extension of SNOMED CT content.

### Column Descriptions
For module dependency files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the module dependency reference set.
*   **`referencedComponentId`**: Identifier of the dependent module.
*   **`sourceEffectiveTime`**: The effective time of the source module.
*   **`targetEffectiveTime`**: The effective time of the target module.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt'

Successfully loaded 'der2_ssRefset_ModuleDependencyFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,sourceEffectiveTime,targetEffectiveTime
0,9200c4da-3189-4dd5-9879-02c9ae118747,20220901,1,5991000124107,900000000000534007,900000000000207008,20220901,20220630
1,9200c4da-3189-4dd5-9879-02c9ae118747,20230301,1,5991000124107,900000000000534007,900000000000207008,20230301,20221231
2,9200c4da-3189-4dd5-9879-02c9ae118747,20230901,1,5991000124107,900000000000534007,900000000000207008,20230901,20230630
3,9200c4da-3189-4dd5-9879-02c9ae118747,20240301,1,5991000124107,900000000000534007,900000000000207008,20240301,20240101
4,9200c4da-3189-4dd5-9879-02c9ae118747,20240901,1,5991000124107,900000000000534007,900000000000207008,20240901,20240701
5,9200c4da-3189-4dd5-9879-02c9ae118747,20250301,1,5991000124107,900000000000534007,900000000000207008,20250301,20250101
6,9200c4da-3189-4dd5-9879-02c9ae118747,20250901,1,5991000124107,900000000000534007,900000000000207008,20250901,20250701
7,9200c4da-3189-4dd5-9879-02c9ae118747,20260301,1,5991000124107,900000000000534007,900000000000207008,20260301,20260101
8,ed482ae6-dff1-42d8-8e39-353b78a60471,20220901,1,5991000124107,900000000000534007,900000000000012004,20220901,20220630
9,ed482ae6-dff1-42d8-8e39-353b78a60471,20230301,1,5991000124107,900000000000534007,900000000000012004,20230301,20221231


## der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt

### Overview
The `der2_sssssssRefset_MRCMDomainFull` file defines the domain of concepts that are constrained by a particular MRCM rule. It specifies which subsets of SNOMED CT concepts are subject to certain modeling rules, ensuring that content development adheres to the established concept model.

### Column Descriptions
For MRCM domain files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the MRCM domain reference set.
*   **`referencedComponentId`**: Identifier of the concept representing the domain.
*   **`domainConstraint`**: The constraint that defines the domain.
*   **`parentDomain`**: The identifier of the parent domain, if applicable.
*   **`proximalPrimitiveConcept`**: The identifier of the closest primitive concept.
*   **`proximalPrimitiveRefinement`**: The refinement expression for the proximal primitive concept.
*   **`domainTemplateForPrecoordination`**: A template for precoordinated concepts.
*   **`domainTemplateForPostcoordination`**: A template for postcoordinated concepts.
*   **`guideURL`**: A URL to a guide or documentation related to the domain.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Full', 'Refset', 'Metadata', 'der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt'

Successfully loaded 'der2_sssssssRefset_MRCMDomainFull_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,domainConstraint,parentDomain,proximalPrimitiveConstraint,proximalPrimitiveRefinement,domainTemplateForPrecoordination,domainTemplateForPostcoordination,guideURL
0,821b5d55-2985-4293-98a9-2959a69b0b3b,20170731,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
1,821b5d55-2985-4293-98a9-2959a69b0b3b,20190731,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
2,821b5d55-2985-4293-98a9-2959a69b0b3b,20200731,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
3,821b5d55-2985-4293-98a9-2959a69b0b3b,20231001,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
4,821b5d55-2985-4293-98a9-2959a69b0b3b,20240501,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
5,821b5d55-2985-4293-98a9-2959a69b0b3b,20250201,1,900000000000012004,723560006,71388002,<< 71388002 |Procedure (procedure)|,NaN,<< 71388002 |Procedure (procedure)|,NaN,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom71388002
6,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20170731,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [+id(<< 12926500...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000
7,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20180131,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [+id(<< 12926500...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000
8,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20190731,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [+id(<< 12926500...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000
9,dcc4c43d-0ff3-448a-b200-fc9c6c3e7a56,20200131,1,900000000000012004,723560006,386053000,<< 386053000 |Evaluation procedure (procedure)|,71388002 |Procedure (procedure)|,<< 71388002 |Procedure (procedure)|,[[1..*]] 260686004 |Method| = [[+id(<< 1292650...,[[+id(<< 71388002 |Procedure (procedure)|)]]: ...,[[+scg(<< 71388002 |Procedure (procedure)|)]]:...,http://snomed.org/dom386053000


## der2_cRefset_LanguageSnapshot-en_US1000124_20260301.txt

### Overview
The `der2_cRefset_LanguageSnapshot` file is the snapshot version of the Language Reference Set. It contains the most current version of each language refset member, specifying the acceptability of descriptions (terms) within a particular language dialect. This file is typically used for day-to-day operations where only the latest active state of components is required.

### Column Descriptions
For language refset snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the reference set itself (e.g., the English language reference set).
*   **`referencedComponentId`**: Identifier of the description that is a member of this refset.
*   **`acceptabilityId`**: Indicates the acceptability of the description for the language (e.g., 900000000000548007 for Preferred, 900000000000549004 for Acceptable).

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Refset', 'Language', 'der2_cRefset_LanguageSnapshot-en_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_cRefset_LanguageSnapshot-en_US1000124_20260301.txt'

Successfully loaded 'der2_cRefset_LanguageSnapshot-en_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,acceptabilityId
0,80000517-8513-5ca0-a44c-dc66f3c3a1c6,20080731,1,900000000000207008,900000000000508004,2743026013,900000000000548007
1,80000755-c5d9-5bd8-bb64-ab8236d240d7,20020131,1,900000000000207008,900000000000509007,2320010,900000000000548007
2,8000095c-e40d-56d2-9432-7f9a716d60d2,20020131,1,900000000000207008,900000000000509007,99175018,900000000000548007
3,80000cf0-bdc8-48e9-b0e1-914e28484bbc,20210731,1,900000000000207008,900000000000508004,4571203018,900000000000549004
4,800012f3-4937-481b-b37c-7862b7073648,20220731,1,900000000000207008,900000000000509007,5071815010,900000000000548007
5,80001355-118d-5beb-a603-48dae9011f64,20020131,1,900000000000207008,900000000000508004,306538015,900000000000549004
6,800016a3-27ca-4e27-9a46-6899408ff2ce,20170131,1,900000000000207008,900000000000508004,3329492019,900000000000549004
7,800023dc-3835-5c28-8a24-3885e165e266,20020131,1,900000000000207008,900000000000509007,381144012,900000000000548007
8,80003e21-1a84-5850-b725-e1a92c6d1fd6,20020131,0,900000000000207008,900000000000509007,108667019,900000000000549004
9,80003eba-0c7c-5dd4-9f9e-a416967cbf05,20150131,1,900000000000207008,900000000000509007,3011884016,900000000000549004


### Snapshot/Refset/Map

## sct2_Concept_Snapshot_US1000124_20260301.txt

### Overview
The `sct2_Concept_Snapshot` file is the snapshot version of the Concept file. It contains the most current details about all active concepts in the terminology. Each concept represents a clinical idea or meaning, and this file is used for day-to-day operations requiring the latest active state of concepts without historical details.

### Column Descriptions
For concept snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the concept.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the concept is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`definitionStatusId`**: Indicates whether the concept is primitive (900000000000074008) or fully defined (900000000000073002).

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Terminology', 'sct2_Concept_Snapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'sct2_Concept_Snapshot_US1000124_20260301.txt'

Successfully loaded 'sct2_Concept_Snapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,definitionStatusId
0,100005,20020131,0,900000000000207008,900000000000074008
1,101009,20020131,1,900000000000207008,900000000000074008
2,102002,20020131,1,900000000000207008,900000000000074008
3,103007,20020131,1,900000000000207008,900000000000074008
4,104001,20020131,1,900000000000207008,900000000000073002
5,105000,20040731,0,900000000000207008,900000000000074008
6,106004,20240301,0,900000000000207008,900000000000074008
7,107008,20020131,1,900000000000207008,900000000000074008
8,108003,20020131,1,900000000000207008,900000000000074008
9,109006,20251101,0,900000000000207008,900000000000074008


### Snapshot/Refset/Language

## sct2_Description_Snapshot-en_US1000124_20260301.txt

### Overview
The `sct2_Description_Snapshot` file is the snapshot version of the Description file. It contains the most current textual descriptions (terms) associated with SNOMED CT concepts, filtered by language. This includes fully specified names, preferred terms, and synonyms for each concept, used for current terminology representations.

### Column Descriptions
For description snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the description.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the description is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`conceptId`**: Identifier of the concept to which this description is attached.
*   **`languageCode`**: Language of the term (e.g., 'en').
*   **`typeId`**: Type of description (e.g., 900000000000003001 for Fully Specified Name, 900000000000013009 for Synonym).
*   **`term`**: The actual textual description.
*   **`caseSignificanceId`**: Indicates how the term's casing should be treated for matching (e.g., 900000000000017005 for entire term case insensitive).

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Terminology', 'sct2_Description_Snapshot-en_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'sct2_Description_Snapshot-en_US1000124_20260301.txt'

Successfully loaded 'sct2_Description_Snapshot-en_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,conceptId,languageCode,typeId,term,caseSignificanceId
0,101013,20170731,1,900000000000207008,126813005,en,900000000000013009,Neoplasm of anterior aspect of epiglottis,900000000000448009
1,102018,20170731,1,900000000000207008,126814004,en,900000000000013009,Neoplasm of junctional region of epiglottis,900000000000448009
2,103011,20170731,1,900000000000207008,126815003,en,900000000000013009,Neoplasm of lateral wall of oropharynx,900000000000448009
3,104017,20170731,1,900000000000207008,126816002,en,900000000000013009,Neoplasm of posterior wall of oropharynx,900000000000448009
4,105016,20170731,1,900000000000207008,126817006,en,900000000000013009,Neoplasm of esophagus,900000000000448009
5,106015,20170731,1,900000000000207008,126818001,en,900000000000013009,Neoplasm of cervical esophagus,900000000000448009
6,107012,20170731,1,900000000000207008,126819009,en,900000000000013009,Neoplasm of thoracic esophagus,900000000000448009
7,108019,20170731,1,900000000000207008,126820003,en,900000000000013009,Neoplasm of abdominal esophagus,900000000000448009
8,110017,20170731,1,900000000000207008,126822006,en,900000000000013009,Neoplasm of middle third of esophagus,900000000000448009
9,111018,20170731,1,900000000000207008,126823001,en,900000000000013009,Neoplasm of lower third of esophagus,900000000000448009


## sct2_Identifier_Snapshot_US1000124_20260301.txt

### Overview
The `sct2_Identifier_Snapshot` file is the snapshot version of the Identifier file. It contains the most current identifiers for SNOMED CT components (concepts, descriptions, and relationships) that are maintained within the system but may not be the primary identifiers. This is useful for tracking external identifiers or alternative identification schemes.

### Column Descriptions
For identifier snapshot files, the columns typically include:
*   **`identifierSchemeId`**: Identifier of the scheme to which the identifier belongs.
*   **`alternateIdentifier`**: The alternative identifier for the component.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the component is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`referencedComponentId`**: Identifier of the SNOMED CT component to which the identifier applies.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Terminology', 'sct2_Identifier_Snapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'sct2_Identifier_Snapshot_US1000124_20260301.txt'

Successfully loaded 'sct2_Identifier_Snapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,alternateIdentifier,effectiveTime,active,moduleId,identifierSchemeId,referencedComponentId


## sct2_RelationshipConcreteValues_Snapshot_US1000124_20260301.txt

### Overview
The `sct2_RelationshipConcreteValues_Snapshot` file is the snapshot version of the Relationship Concrete Values file. It contains the most current relationships where the target is a concrete value (e.g., a number, string, or boolean) rather than another concept. This is crucial for representing quantitative or qualitative attributes of concepts directly.

### Column Descriptions
For relationship concrete values snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the relationship.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the relationship is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`sourceId`**: Identifier of the concept that is the source of the relationship.
*   **`value`**: The concrete value (e.g., '10', 'true', 'some text').
*   **`relationshipGroup`**: An integer indicating a group of relationships.
*   **`typeId`**: Identifier of the concept representing the type of relationship.
*   **`characteristicTypeId`**: Identifier of the concept representing the characteristic type of the relationship.
*   **`modifierId`**: Identifier of the concept representing the modifier of the relationship (e.g., 'existential', 'universal').

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Terminology', 'sct2_RelationshipConcreteValues_Snapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'sct2_RelationshipConcreteValues_Snapshot_US1000124_20260301.txt'

Successfully loaded 'sct2_RelationshipConcreteValues_Snapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,sourceId,value,relationshipGroup,typeId,characteristicTypeId,modifierId
0,13830203029,20210731,1,900000000000207008,830045007,#3,0,1142139005,900000000000011006,900000000000451002
1,13830204024,20210731,1,900000000000207008,830064001,#3,0,1142139005,900000000000011006,900000000000451002
2,13830205020,20210731,1,900000000000207008,830066004,#3,0,1142139005,900000000000011006,900000000000451002
3,13830206021,20210731,1,900000000000207008,830108003,#1,0,1142139005,900000000000011006,900000000000451002
4,13830207028,20210731,1,900000000000207008,830110001,#1,0,1142139005,900000000000011006,900000000000451002
5,13830208022,20210731,1,900000000000207008,830208008,#1,0,1142139005,900000000000011006,900000000000451002
6,13830209025,20210731,1,900000000000207008,830210005,#1,0,1142139005,900000000000011006,900000000000451002
7,13830210024,20210731,1,900000000000207008,830203004,#1,0,1142139005,900000000000011006,900000000000451002
8,13830211023,20210731,1,900000000000207008,830204005,#1,0,1142139005,900000000000011006,900000000000451002
9,13830212027,20210731,1,900000000000207008,830205006,#1,0,1142139005,900000000000011006,900000000000451002


## sct2_Relationship_Snapshot_US1000124_20260301.txt

### Overview
The `sct2_Relationship_Snapshot` file is the snapshot version of the Relationship file. It contains the most current relationships between SNOMED CT concepts, defining the hierarchy and other semantic connections. These relationships are fundamental for navigating and understanding the structure of SNOMED CT.

### Column Descriptions
For relationship snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the relationship.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the relationship is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`sourceId`**: Identifier of the concept that is the source of the relationship.
*   **`destinationId`**: Identifier of the concept that is the target of the relationship.
*   **`relationshipGroup`**: An integer indicating a group of relationships.
*   **`typeId`**: Identifier of the concept representing the type of relationship (e.g., 'Is a', 'Associated morphology').
*   **`characteristicTypeId`**: Identifier of the concept representing the characteristic type of the relationship (e.g., 'Inferred relationship', 'Stated relationship').
*   **`modifierId`**: Identifier of the concept representing the modifier of the relationship (e.g., 'Existential restriction', 'Universal restriction').

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Terminology', 'sct2_Relationship_Snapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'sct2_Relationship_Snapshot_US1000124_20260301.txt'

Successfully loaded 'sct2_Relationship_Snapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,sourceId,destinationId,relationshipGroup,typeId,characteristicTypeId,modifierId
0,100022,20090731,0,900000000000207008,100000000,102272007,0,116680003,900000000000011006,900000000000451002
1,101021,20020131,1,900000000000207008,10000006,29857009,0,116680003,900000000000011006,900000000000451002
2,102025,20020131,1,900000000000207008,10000006,9972008,0,116680003,900000000000011006,900000000000451002
3,103024,20030131,0,900000000000207008,1000004,19130008,0,116680003,900000000000011006,900000000000451002
4,104029,20090731,0,900000000000207008,100001001,102272007,0,116680003,900000000000011006,900000000000451002
5,105028,20090731,0,900000000000207008,100002008,102272007,0,116680003,900000000000011006,900000000000451002
6,106027,20090731,0,900000000000207008,100003003,102272007,0,116680003,900000000000011006,900000000000451002
7,107020,20090731,0,900000000000207008,100004009,102272007,0,116680003,900000000000011006,900000000000451002
8,109023,20090731,0,900000000000207008,100005005,102272007,0,116680003,900000000000011006,900000000000451002
9,110029,20090731,0,900000000000207008,100006006,102272007,0,116680003,900000000000011006,900000000000451002


## sct2_sRefset_OWLExpressionSnapshot_US1000124_20260301.txt

### Overview
The `sct2_sRefset_OWLExpressionSnapshot` file is the snapshot version of the OWL Expression Reference Set. It contains the most current OWL (Web Ontology Language) expressions that define SNOMED CT concepts and their properties formally. This file is used for advanced ontological reasoning and interoperability with OWL-based systems.

### Column Descriptions
For OWL expression refset snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the reference set itself.
*   **`referencedComponentId`**: Identifier of the SNOMED CT concept to which the OWL expression applies.
*   **`owlExpression`**: The OWL expression string.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Terminology', 'sct2_sRefset_OWLExpressionSnapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'sct2_sRefset_OWLExpressionSnapshot_US1000124_20260301.txt'

Successfully loaded 'sct2_sRefset_OWLExpressionSnapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,owlExpression
0,80001735-381a-4c86-a986-a6ebd875f6c7,20190731,1,900000000000207008,733073007,42061009,SubClassOf(:42061009 :398334008)
1,80002779-6efa-491f-88d3-8a393898bbe4,20190731,1,900000000000207008,733073007,239604004,SubClassOf(:239604004 ObjectIntersectionOf(:26...
2,80004459-1608-4ec1-9d41-ae994cd7f6a9,20190731,1,900000000000207008,733073007,283910009,SubClassOf(:283910009 :283904009)
3,80005cdc-07bf-41b8-9e90-47393b071a6a,20190731,1,900000000000207008,733073007,721657003,EquivalentClasses(:721657003 ObjectIntersectio...
4,8000d644-fed0-498d-8882-3e69d3f964d1,20190731,1,900000000000207008,733073007,87885009,SubClassOf(:87885009 :2090006)
5,80012aef-d46d-4fbb-b0b7-0e10e4f99d77,20231201,1,900000000000207008,733073007,303394007,EquivalentClasses(:303394007 ObjectIntersectio...
6,8001566d-6275-4931-a567-f6fd7cd48bca,20190731,1,900000000000207008,733073007,84132007,SubClassOf(:84132007 :429240000)
7,80015875-73ae-4065-b2e5-c3c0e87a9214,20250301,1,900000000000207008,733073007,1144573006,EquivalentClasses(:1144573006 ObjectIntersecti...
8,80017b56-d222-4920-8585-0d34d159c755,20210930,1,900000000000207008,733073007,1162314004,EquivalentClasses(:1162314004 ObjectIntersecti...
9,8001bb88-31b2-4c28-8f80-346ec62d5c1a,20190731,1,900000000000207008,733073007,43205001,SubClassOf(:43205001 :112394008)


## sct2_StatedRelationship_Snapshot_US1000124_20260301.txt

### Overview
The `sct2_StatedRelationship_Snapshot` file is the snapshot version of the Stated Relationship file. It contains the most current explicitly asserted (stated) relationships between SNOMED CT concepts. These relationships represent the direct knowledge entered by modelers, in contrast to inferred relationships.

### Column Descriptions
For stated relationship snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the relationship.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the relationship is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`sourceId`**: Identifier of the concept that is the source of the relationship.
*   **`destinationId`**: Identifier of the concept that is the target of the relationship.
*   **`relationshipGroup`**: An integer indicating a group of relationships.
*   **`typeId`**: Identifier of the concept representing the type of relationship (e.g., 'Is a', 'Associated morphology').
*   **`characteristicTypeId`**: Identifier of the concept representing the characteristic type of the relationship (always 'Stated relationship' for this file).
*   **`modifierId`**: Identifier of the concept representing the modifier of the relationship.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Terminology', 'sct2_StatedRelationship_Snapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'sct2_StatedRelationship_Snapshot_US1000124_20260301.txt'

Successfully loaded 'sct2_StatedRelationship_Snapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,sourceId,destinationId,relationshipGroup,typeId,characteristicTypeId,modifierId
0,3187444026,20160131,0,900000000000207008,425630003,400195000,0,42752001,900000000000010007,900000000000451002
1,3192499027,20140131,0,900000000000207008,425630003,105590001,0,246075003,900000000000010007,900000000000451002
2,3574321020,20160131,0,900000000000207008,425630003,111189002,0,116680003,900000000000010007,900000000000451002
3,3829433029,20190731,0,900000000000207008,102977005,102976001,0,116680003,900000000000010007,900000000000451002
4,3829434024,20190731,0,900000000000207008,413337008,306751006,0,116680003,900000000000010007,900000000000451002
5,3829435020,20190731,0,900000000000207008,103085008,72909000,0,116680003,900000000000010007,900000000000451002
6,3829436021,20190731,0,900000000000207008,103085008,259648002,0,116680003,900000000000010007,900000000000451002
7,3829437028,20190731,0,900000000000207008,103142002,40992002,0,116680003,900000000000010007,900000000000451002
8,3829438022,20190731,0,900000000000207008,103143007,40992002,0,116680003,900000000000010007,900000000000451002
9,3829439025,20190731,0,900000000000207008,103144001,40992002,0,116680003,900000000000010007,900000000000451002


## sct2_TextDefinition_Snapshot-en_US1000124_20260301.txt

### Overview
The `sct2_TextDefinition_Snapshot` file is the snapshot version of the Text Definition file. It contains the most current human-readable definitions for SNOMED CT concepts, provided as text. These definitions are essential for understanding the precise meaning of concepts within the terminology.

### Column Descriptions
For text definition snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the text definition.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the text definition is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`conceptId`**: Identifier of the concept being defined.
*   **`languageCode`**: Language of the definition (e.g., 'en').
*   **`typeId`**: Type of definition (e.g., 900000000000550004 for textual definition).
*   **`term`**: The actual text of the definition.
*   **`caseSignificanceId`**: Indicates how the term's casing should be treated.
*   **`textDefinitionId`**: The identifier of the concept that represents the text definition.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Terminology', 'sct2_TextDefinition_Snapshot-en_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'sct2_TextDefinition_Snapshot-en_US1000124_20260301.txt'

Successfully loaded 'sct2_TextDefinition_Snapshot-en_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,conceptId,languageCode,typeId,term,caseSignificanceId
0,2884452019,20190731,0,900000000000207008,410016009,en,900000000000550004,A decrease in lower leg circumference due to r...,900000000000017005
1,2884453012,20050731,1,900000000000207008,416118004,en,900000000000550004,Introduction of a substance to the body,900000000000017005
2,2884454018,20030731,1,900000000000207008,125097000,en,900000000000550004,Domestic goat,900000000000017005
3,2884455017,20110131,0,900000000000207008,125099002,en,900000000000550004,Domestic sheep species,900000000000017005
4,2884456016,20030731,1,900000000000207008,122868007,en,900000000000550004,An implantation of a staple,900000000000017005
5,2884457013,20100731,0,900000000000207008,125085001,en,900000000000550004,Equus subspecies,900000000000017005
6,2884458015,20030731,1,900000000000207008,125671007,en,900000000000550004,"Disruption of continuity of tissue, not necess...",900000000000017005
7,2884459011,20030731,1,900000000000207008,127062003,en,900000000000550004,Peripheral blood red cell count above the norm...,900000000000017005
8,2884460018,20200131,0,900000000000207008,2504000,en,900000000000550004,An autonomic plexus that is a subdivision of t...,900000000000017005
9,2884461019,20200131,0,900000000000207008,1431002,en,900000000000550004,"The act or operation of holding, suturing, or ...",900000000000017005


## der2_iisssccRefset_ExtendedMapSnapshot_US1000124_20260301.txt

### Overview
The `der2_iisssccRefset_ExtendedMapSnapshot` file is the snapshot version of the Extended Map Reference Set. It contains the most current version of each extended map refset member, used for mapping SNOMED CT concepts to external terminologies with additional attributes. This snapshot file is ideal for applications requiring the latest mapping information without historical details.

### Column Descriptions
For extended map refset snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the reference set itself.
*   **`referencedComponentId`**: Identifier of the SNOMED CT concept being mapped.
*   **`mapGroup`**: A group number indicating related mapping entries.
*   **`mapPriority`**: An integer indicating the priority of the map target within a map group.
*   **`mapRule`**: A rule that defines the conditions under which a map target is applicable.
*   **`mapAdvice`**: Text providing advice or instruction about the use of the map.
*   **`mapTarget`**: The identifier of the target concept in the external terminology.
*   **`correlationId`**: An identifier representing the degree of correlation between source and target concepts.
*   **`mapCategoryId`**: An identifier for the category of the map (e.g., fully specified, partial, ambiguous).

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Refset', 'Map', 'der2_iisssccRefset_ExtendedMapSnapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_iisssccRefset_ExtendedMapSnapshot_US1000124_20260301.txt'

Successfully loaded 'der2_iisssccRefset_ExtendedMapSnapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,mapGroup,mapPriority,mapRule,mapAdvice,mapTarget,correlationId,mapCategoryId
0,80004cdf-a114-59c8-88a2-b6f0283acf4f,20180301,1,5991000124107,6011000124106,128041000119107,1,1,IFA 721617001 | Primary adenocarcinoma of lowe...,IF PRIMARY ADENOCARCINOMA OF LOWER THIRD OF ES...,K22.70,447561005,447639009
1,80005aeb-477c-53dc-9a5c-ce723ca264cb,20150731,1,449080006,447562003,254153009,1,1,TRUE,ALWAYS Q79.8,Q79.8,447561005,447637006
2,80007b64-5a60-5556-88ce-22ef540a3ea5,20250301,0,5991000124107,6011000124106,281801000009108,2,10,OTHERWISE TRUE,ALWAYS B96.89,B96.89,447561005,447637006
3,80007e6a-7408-5b87-a1ec-70b212811410,20190731,1,449080006,447562003,16623961000119100,1,1,TRUE,ALWAYS D61.1,D61.1,447561005,447637006
4,80009454-5531-5f78-b7c9-d288f2346d83,20190131,0,449080006,447562003,301327002,1,1,TRUE,MAP SOURCE CONCEPT CANNOT BE CLASSIFIED WITH A...,NaN,447561005,447638001
5,8000a5af-6962-5385-9227-4038d1f7b237,20150731,1,449080006,447562003,246951001,1,1,TRUE,ALWAYS H10.8,H10.8,447561005,447637006
6,8000ab4e-1417-5c6e-bb3a-9b7ae751bc03,20250301,1,5991000124107,6011000124106,10996811000119103,1,2,IFA 53911000087103 | History of bilateral inde...,IF HISTORY OF BILATERAL INDEX FINGER AMPUTATIO...,Z89.021,447561005,447639009
7,8000fff7-d0ac-5ad4-82eb-f8e0a37ebfc7,20190301,1,5991000124107,6011000124106,292278006,1,1,TRUE,ALWAYS T43.595? | CONSIDER ADDITIONAL CODE TO ...,T43.595?,447561005,447637006
8,80010c58-f11f-572f-98f9-852cd301d5c0,20200131,0,449080006,447562003,291710005,2,1,TRUE,ALWAYS X44 | POSSIBLE REQUIREMENT FOR PLACE OF...,X44,447561005,447637006
9,80011508-c40d-5086-a12a-bcc1f5a4cea2,20230301,0,5991000124107,6011000124106,93764002,1,2,IFA 231833004 | Sebaceous adenocarcinoma of ey...,IF SEBACEOUS ADENOCARCINOMA OF EYELID CHOOSE C...,C44.191,447561005,447639009


## der2_sRefset_SimpleMapSnapshot_US1000124_20260301.txt

### Overview
The `der2_sRefset_SimpleMapSnapshot` file is the snapshot version of the Simple Map Reference Set. It contains the most current version of each simple map refset member, providing straightforward one-to-one mappings from SNOMED CT concepts to concepts in other terminologies or classifications. This snapshot file is used when only the latest, active mappings are needed.

### Column Descriptions
For simple map refset snapshot files, the columns typically include:
*   **`id`**: Unique identifier for the refset member.
*   **`effectiveTime`**: Date and time when the component version became active.
*   **`active`**: Indicates if the refset member is active (1) or inactive (0).
*   **`moduleId`**: Identifier of the module to which the component belongs.
*   **`refsetId`**: Identifier of the reference set itself.
*   **`referencedComponentId`**: Identifier of the SNOMED CT concept being mapped.
*   **`mapTarget`**: The identifier of the target concept in the external terminology.

In [ ]:
file_path = os.path.join(snomed_base_path, 'Snapshot', 'Refset', 'Map', 'der2_sRefset_SimpleMapSnapshot_US1000124_20260301.txt')

print(f"### Reading '{os.path.basename(file_path)}'\n")
try:
    df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip', nrows=10)
    print(f"Successfully loaded '{os.path.basename(file_path)}'. Displaying the first 10 rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading '{os.path.basename(file_path)}': {e}")

### Reading 'der2_sRefset_SimpleMapSnapshot_US1000124_20260301.txt'

Successfully loaded 'der2_sRefset_SimpleMapSnapshot_US1000124_20260301.txt'. Displaying the first 10 rows:


,id,effectiveTime,active,moduleId,refsetId,referencedComponentId,mapTarget
0,80001267-7451-550a-82f0-92cc3bdfe890,20020131,1,900000000000207008,900000000000497000,154938001,.E4D4
1,80001782-d79c-5b33-8679-c0c62beef6da,20020131,1,900000000000207008,900000000000497000,138614002,.13gX
2,8000241a-ed32-4339-876b-05fee677bda3,20180131,1,900000000000207008,900000000000497000,735755000,XUyL5
3,80002a2a-412f-59a4-b07c-6f194709c556,20020131,1,900000000000207008,900000000000497000,238194001,X40Ze
4,80004caa-f9ed-5ef8-a9fb-6c9e89e0b89d,20020131,1,900000000000207008,900000000000497000,181522009,7N72Y
5,8000501c-e5f1-5df2-91b8-d2360661e55c,20050731,1,900000000000207008,900000000000497000,416460006,XUd3b
6,800061cc-1d06-4048-874d-2d5eff76a653,20180131,1,900000000000207008,900000000000497000,16064651000119108,XUyFc
7,80009e45-0c82-4285-9ba4-b6de7e29d7e9,20190731,1,900000000000207008,900000000000497000,10823771000119109,XVAEl
8,8000b9de-9288-5bfb-905b-adb911e6cef5,20020131,1,900000000000207008,900000000000497000,317571000,za1CK
9,8000baeb-e1a1-50e8-b28f-7a687d3cf1c5,20020131,1,900000000000207008,446608001,2681003,C47.5


### Snapshot/Refset/Content